## 本机环境

在Jupyter选择 `FNSPID FinBERT (project)` 内核；如果看不到该名称，在编辑器选择项目的 `work/finbert_env/bin/python` 解释器。

模型缓存保存在项目的work/huggingface_cache。下面设置已固定本次通过验证的模型commit。不要在旧内核中先导入transformers再执行缓存设置；应重启并从头运行。

小样本验证结果见finbert_install_verification.json；这不代表金融新闻情感识别准确率已获验证。

In [12]:
import subprocess
from pathlib import Path

python_path = Path(
    "/Users/jujube/Documents/Codex/2026-11-04/"
    "wps-km6312-ddl-11-19-https/work/finbert_env/bin/python"
)

subprocess.run(
    [
        str(python_path), "-m", "ipykernel", "install",
        "--user",
        "--name", "fnspid-finbert",
        "--display-name", "FNSPID FinBERT (project)"
    ],
    check=True
)

Installed kernelspec fnspid-finbert in /Users/jujube/Library/Jupyter/kernels/fnspid-finbert


CompletedProcess(args=['/Users/jujube/Documents/Codex/2026-11-04/wps-km6312-ddl-11-19-https/work/finbert_env/bin/python', '-m', 'ipykernel', 'install', '--user', '--name', 'fnspid-finbert', '--display-name', 'FNSPID FinBERT (project)'], returncode=0)

In [13]:
from pathlib import Path
import os, sys
PROJECT_ROOT_LOCAL = Path('/Users/jujube/Documents/Codex/2026-11-04/wps-km6312-ddl-11-19-https')
assert PROJECT_ROOT_LOCAL.exists(), '在其他电脑运行时，请修改PROJECT_ROOT_LOCAL'
os.chdir(PROJECT_ROOT_LOCAL)
os.environ['HF_HOME'] = str(PROJECT_ROOT_LOCAL/'work/huggingface_cache')
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Python:', sys.executable)
import torch, transformers
print('torch:',torch.__version__,'transformers:',transformers.__version__)


Python: /Users/jujube/Documents/Codex/2026-11-04/wps-km6312-ddl-11-19-https/work/finbert_env/bin/python
torch: 2.15.0 transformers: 5.22.2


## 基础阶段：FNSPID数据集划分与样本统计

In [5]:
from pathlib import Path
import csv, io, json, sqlite3, zipfile, math, datetime, bisect, time, os
from collections import Counter

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'outputs':
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'work/fnspid').is_dir(), '请将PROJECT_ROOT改为项目根目录'
os.chdir(PROJECT_ROOT)
base = Path('work/fnspid')
out = Path('outputs')
out.mkdir(exist_ok=True)
expected_sizes = {'All_external.csv': 5731397037, 'nasdaq_exteral_data.csv': 23232979597, 'full_history.zip': 589525596}
for name, size in expected_sizes.items():
    path = base / name
    assert path.is_file() and path.stat().st_size == size, f'文件缺失或大小不符：{name}'
    print(name, f'{path.stat().st_size:,} bytes')


All_external.csv 5,731,397,037 bytes
nasdaq_exteral_data.csv 23,232,979,597 bytes
full_history.zip 589,525,596 bytes


## 冻结成分股名单

In [6]:
SP500_SNAPSHOT = 'Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded\nMMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902\nAOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916\nABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888\nABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)\nACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989\nADBE,Adobe Inc.,Information Technology,Application Software,"San Jose, California",1997-05-05,796343,1982\nAMD,Advanced Micro Devices,Information Technology,Semiconductors,"Santa Clara, California",2017-03-20,2488,1969\nAES,AES Corporation,Utilities,Independent Power Producers & Energy Traders,"Arlington, Virginia",1998-10-02,874761,1981\nAFL,Aflac,Financials,Life & Health Insurance,"Columbus, Georgia",1999-05-28,4977,1955\nA,Agilent Technologies,Health Care,Life Sciences Tools & Services,"Santa Clara, California",2000-06-05,1090872,1999\nAPD,Air Products,Materials,Industrial Gases,"Upper Macungie Township, Pennsylvania",1985-04-30,2969,1940\nABNB,Airbnb,Consumer Discretionary,"Hotels, Resorts & Cruise Lines","San Francisco, California",2023-09-18,1559720,2008\nAKAM,Akamai Technologies,Information Technology,Internet Services & Infrastructure,"Cambridge, Massachusetts",2007-07-12,1086222,1998\nALB,Albemarle Corporation,Materials,Specialty Chemicals,"Charlotte, North Carolina",2016-07-01,915913,1994\nARE,Alexandria Real Estate Equities,Real Estate,Office REITs,"Pasadena, California",2017-03-20,1035443,1994\nALGN,Align Technology,Health Care,Health Care Supplies,"Tempe, Arizona",2017-06-19,1097149,1997\nALLE,Allegion,Industrials,Building Products,"Dublin, Ireland",2013-12-02,1579241,1908\nLNT,Alliant Energy,Utilities,Electric Utilities,"Madison, Wisconsin",2016-07-01,352541,1917\nALL,Allstate,Financials,Property & Casualty Insurance,"Northbrook, Illinois",1995-07-13,899051,1931\nGOOGL,Alphabet Inc. (Class A),Communication Services,Interactive Media & Services,"Mountain View, California",2006-04-03,1652044,1998\nGOOG,Alphabet Inc. (Class C),Communication Services,Interactive Media & Services,"Mountain View, California",2014-04-03,1652044,1998\nMO,Altria,Consumer Staples,Tobacco,"Richmond, Virginia",1957-03-04,764180,1985\nAMZN,Amazon,Consumer Discretionary,Broadline Retail,"Seattle, Washington",2005-11-18,1018724,1994\nAMCR,Amcor,Materials,Paper & Plastic Packaging Products & Materials,"Warmley, Bristol, United Kingdom",2019-06-07,1748790,2019 (1860)\nAEE,Ameren,Utilities,Multi-Utilities,"St. Louis, Missouri",1991-09-19,1002910,1902\nAEP,American Electric Power,Utilities,Electric Utilities,"Columbus, Ohio",1957-03-04,4904,1906\nAXP,American Express,Financials,Consumer Finance,"New York City, New York",1976-06-30,4962,1850\nAIG,American International Group,Financials,Multi-line Insurance,"New York City, New York",1980-03-31,5272,1919\nAMT,American Tower,Real Estate,Telecom Tower REITs,"Boston, Massachusetts",2007-11-19,1053507,1995\nAWK,American Water Works,Utilities,Water Utilities,"Camden, New Jersey",2016-03-04,1410636,1886\nAMP,Ameriprise Financial,Financials,Asset Management & Custody Banks,"Minneapolis, Minnesota",2005-10-03,820027,1894\nAME,Ametek,Industrials,Electrical Components & Equipment,"Berwyn, Pennsylvania",2013-09-23,1037868,1930\nAMGN,Amgen,Health Care,Biotechnology,"Thousand Oaks, California",1992-01-02,318154,1980\nAPH,Amphenol,Information Technology,Electronic Components,"Wallingford, Connecticut",2008-09-30,820313,1932\nADI,Analog Devices,Information Technology,Semiconductors,"Wilmington, Massachusetts",1999-10-12,6281,1965\nAON,Aon plc,Financials,Insurance Brokers,"London, United Kingdom",1996-04-23,315293,1982 (1919)\nAPA,APA Corporation,Energy,Oil & Gas Exploration & Production,"Houston, Texas",1997-07-28,1841666,1954\nAPO,Apollo Global Management,Financials,Asset Management & Custody Banks,"New York City, New York",2024-12-23,1858681,1990\nAAPL,Apple Inc.,Information Technology,"Technology Hardware, Storage & Peripherals","Cupertino, California",1982-11-30,320193,1977\nAMAT,Applied Materials,Information Technology,Semiconductor Materials & Equipment,"Santa Clara, California",1995-03-16,6951,1967\nAPP,AppLovin,Communication Services,Advertising,"Palo Alto, California",2025-09-22,1751008,2012\nAPTV,Aptiv,Consumer Discretionary,Automotive Parts & Equipment,"Schaffhausen, Switzerland",2012-12-24,1521332,1994\nACGL,Arch Capital Group,Financials,Property & Casualty Insurance,"Hamilton, Bermuda",2022-11-01,947484,1995\nADM,Archer Daniels Midland,Consumer Staples,Agricultural Products & Services,"Chicago, Illinois",1957-03-04,7084,1902\nARES,Ares Management,Financials,Asset Management & Custody Banks,"Los Angeles, California",2025-12-11,1176948,1977\nANET,Arista Networks,Information Technology,Communications Equipment,"Santa Clara, California",2018-08-28,1596532,2004\nAJG,Arthur J. Gallagher & Co.,Financials,Insurance Brokers,"Rolling Meadows, Illinois",2016-05-31,354190,1927\nAIZ,Assurant,Financials,Multi-line Insurance,"Atlanta, Georgia",2007-04-10,1267238,1892\nT,AT&T,Communication Services,Integrated Telecommunication Services,"Dallas, Texas",1983-11-30,732717,1983 (1885)\nATO,Atmos Energy,Utilities,Gas Utilities,"Dallas, Texas",2019-02-15,731802,1906\nADSK,Autodesk,Information Technology,Application Software,"San Francisco, California",1989-12-01,769397,1982\nADP,Automatic Data Processing,Industrials,Human Resource & Employment Services,"Roseland, New Jersey",1981-03-31,8670,1949\nAZO,AutoZone,Consumer Discretionary,Automotive Retail,"Memphis, Tennessee",1997-01-02,866787,1979\nAVY,Avery Dennison,Materials,Paper & Plastic Packaging Products & Materials,"Mentor, Ohio",1987-12-31,8818,1935\nAXON,Axon Enterprise,Industrials,Aerospace & Defense,"Scottsdale, Arizona",2023-05-04,1069183,1993\nBKR,Baker Hughes,Energy,Oil & Gas Equipment & Services,"Houston, Texas",2017-07-07,1701605,2017\nBALL,Ball Corporation,Materials,"Metal, Glass & Plastic Containers","Broomfield, Colorado",1984-10-31,9389,1880\nBAC,Bank of America,Financials,Diversified Banks,"Charlotte, North Carolina",1976-06-30,70858,1998 (1923 / 1874)\nBAX,Baxter International,Health Care,Health Care Equipment,"Deerfield, Illinois",1972-09-30,10456,1931\nBDX,Becton Dickinson,Health Care,Health Care Equipment,"Franklin Lakes, New Jersey",1972-09-30,10795,1897\nBRK.B,Berkshire Hathaway,Financials,Multi-Sector Holdings,"Omaha, Nebraska",2010-02-16,1067983,1839\nBBY,Best Buy,Consumer Discretionary,Computer & Electronics Retail,"Richfield, Minnesota",1999-06-29,764478,1966\nTECH,Bio-Techne,Health Care,Life Sciences Tools & Services,"Minneapolis, Minnesota",2021-08-30,842023,1976\nBIIB,Biogen,Health Care,Biotechnology,"Cambridge, Massachusetts",2003-11-13,875045,1978\nBLK,BlackRock,Financials,Asset Management & Custody Banks,"New York City, New York",2011-04-04,2012383,1988\nBX,Blackstone Inc.,Financials,Asset Management & Custody Banks,"New York City, New York",2023-09-18,1393818,1985\nXYZ,"Block, Inc.",Financials,Transaction & Payment Processing Services,none,2025-07-23,1512673,2009\nBE,Bloom Energy,Industrials,Electrical Components & Equipment,"San Jose, California",2026-09-21,1664703,2001\nBNY,BNY Mellon,Financials,Asset Management & Custody Banks,"New York City, New York",1995-03-31,1390777,2007 (1784)\nBA,Boeing,Industrials,Aerospace & Defense,"Arlington, Virginia",1957-03-04,12927,1916\nBKNG,Booking Holdings,Consumer Discretionary,"Hotels, Resorts & Cruise Lines","Norwalk, Connecticut",2009-11-06,1075531,1996\nBSX,Boston Scientific,Health Care,Health Care Equipment,"Marlborough, Massachusetts",1995-02-24,885725,1979\nBMY,Bristol Myers Squibb,Health Care,Pharmaceuticals,"New York City, New York",1957-03-04,14272,1989 (1887)\nAVGO,Broadcom,Information Technology,Semiconductors,"Palo Alto, California",2014-05-08,1730168,1961\nBR,Broadridge Financial Solutions,Industrials,Data Processing & Outsourced Services,"Lake Success, New York",2018-06-18,1383312,1962\nBRO,Brown & Brown,Financials,Insurance Brokers,"Daytona Beach, Florida",2021-09-20,79282,1939\nBF.B,Brown–Forman,Consumer Staples,Distillers & Vintners,"Louisville, Kentucky",1982-10-31,14693,1870\nBG,Bunge Global,Consumer Staples,Agricultural Products & Services,"Chesterfield, Missouri",2023-03-15,1996862,1818\nBXP,"BXP, Inc.",Real Estate,Office REITs,"Boston, Massachusetts",2006-04-03,1037540,1970\nCHRW,C.H. Robinson,Industrials,Air Freight & Logistics,"Eden Prairie, Minnesota",2007-03-02,1043277,1905\nCDNS,Cadence Design Systems,Information Technology,Application Software,"San Jose, California",2017-09-18,813672,1988\nCPT,Camden Property Trust,Real Estate,Multi-Family Residential REITs,"Houston, Texas",2022-04-04,906345,1981\nCOF,Capital One,Financials,Consumer Finance,"Tysons Corner, Virginia",1998-07-01,927628,1994\nCAH,Cardinal Health,Health Care,Health Care Distributors,"Dublin, Ohio",1997-05-27,721371,1971\nCCL,Carnival Corporation,Consumer Discretionary,"Hotels, Resorts & Cruise Lines","Miami, Florida",1998-12-22,815097,1972\nCARR,Carrier Global,Industrials,Building Products,"Palm Beach Gardens, Florida",2020-04-03,1783180,"2020 (1915, United Technologies spinoff)"\nCVNA,Carvana,Consumer Discretionary,Automotive Retail,"Tempe, Arizona",2025-12-22,1690820,2012\nCASY,Casey\'s,Consumer Staples,Food Retail,"Ankeny, Iowa",2026-04-09,726958,1967\nCAT,Caterpillar Inc.,Industrials,Construction Machinery & Heavy Transportation Equipment,"Irving, Texas",1957-03-04,18230,1925\nCBOE,Cboe Global Markets,Financials,Financial Exchanges & Data,"Chicago, Illinois",2017-03-01,1374310,1973\nCBRE,CBRE Group,Real Estate,Real Estate Services,"Dallas, Texas",2006-11-10,1138118,1906\nCDW,CDW Corporation,Information Technology,Technology Distributors,"Vernon Hills, Illinois",2019-09-23,1402057,1984\nCOR,Cencora,Health Care,Health Care Distributors,"Conshohocken, Pennsylvania",2001-08-30,1140859,1985\nCNC,Centene Corporation,Health Care,Managed Health Care,"St. Louis, Missouri",2016-03-30,1071739,1984\nCNP,CenterPoint Energy,Utilities,Multi-Utilities,"Houston, Texas",1985-07-31,1130310,1882\nCF,CF Industries,Materials,Fertilizers & Agricultural Chemicals,"Deerfield, Illinois",2008-08-27,1324404,1946\nCRL,Charles River Laboratories,Health Care,Life Sciences Tools & Services,"Wilmington, Massachusetts",2021-05-14,1100682,1947\nSCHW,Charles Schwab Corporation,Financials,Investment Banking & Brokerage,"Westlake, Texas",1997-06-02,316709,1971\nCHTR,Charter Communications,Communication Services,Cable & Satellite,"Stamford, Connecticut",2016-09-08,1091667,1993\nCVX,Chevron Corporation,Energy,Integrated Oil & Gas,"Houston, Texas",1957-03-04,93410,1879\nCMG,Chipotle Mexican Grill,Consumer Discretionary,Restaurants,"Newport Beach, California",2011-04-28,1058090,1993\nCB,Chubb Limited,Financials,Property & Casualty Insurance,"Zurich, Switzerland",2010-07-15,896159,1985\nCHD,Church & Dwight,Consumer Staples,Household Products,"Ewing, New Jersey",2015-12-29,313927,1847\nCIEN,Ciena,Information Technology,Communications Equipment,"Hanover, Maryland",2026-02-09,936395,1992\nCI,Cigna,Health Care,Health Care Services,"Bloomfield, Connecticut",1976-06-30,1739940,1982\nCINF,Cincinnati Financial,Financials,Property & Casualty Insurance,"Fairfield, Ohio",1997-12-18,20286,1950\nCTAS,Cintas,Industrials,Diversified Support Services,"Mason, Ohio",2001-03-01,723254,1929\nCSCO,Cisco,Information Technology,Communications Equipment,"San Jose, California",1993-12-01,858877,1984\nC,Citigroup,Financials,Diversified Banks,"New York City, New York",1988-05-31,831001,1998\nCFG,Citizens Financial Group,Financials,Regional Banks,"Providence, Rhode Island",2016-01-29,759944,1828\nCLX,Clorox,Consumer Staples,Household Products,"Oakland, California",1969-03-31,21076,1913\nCME,CME Group,Financials,Financial Exchanges & Data,"Chicago, Illinois",2006-08-11,1156375,1848\nCMS,CMS Energy,Utilities,Multi-Utilities,"Jackson, Michigan",1957-03-04,811156,1886\nKO,Coca-Cola Company (The),Consumer Staples,Soft Drinks & Non-alcoholic Beverages,"Atlanta, Georgia",1957-03-04,21344,1886\nCTSH,Cognizant,Information Technology,IT Consulting & Other Services,"Teaneck, New Jersey",2006-11-17,1058290,1994\nCOHR,Coherent Corp.,Information Technology,Electronic Components,"Saxonburg, Pennsylvania",2026-03-23,820318,1971\nCOIN,Coinbase,Financials,Financial Exchanges & Data,"New York City, New York",2025-05-19,1679788,2012\nCL,Colgate-Palmolive,Consumer Staples,Household Products,"New York City, New York",1957-03-04,21665,1806\nCMCSA,Comcast,Communication Services,Cable & Satellite,"Philadelphia, Pennsylvania",2002-11-19,1166691,1963\nFIX,Comfort Systems USA,Industrials,Construction & Engineering,"Houston, Texas",2025-12-22,1035983,1997\nCOP,ConocoPhillips,Energy,Oil & Gas Exploration & Production,"Houston, Texas",1957-03-04,1163165,2002\nED,Consolidated Edison,Utilities,Multi-Utilities,"New York City, New York",1957-03-04,1047862,1823\nSTZ,Constellation Brands,Consumer Staples,Distillers & Vintners,"Rochester, New York",2005-07-01,16918,1945\nCEG,Constellation Energy,Utilities,Electric Utilities,"Baltimore, Maryland",2022-02-02,1868275,1999\nCOO,Cooper Companies (The),Health Care,Health Care Supplies,"San Ramon, California",2016-09-23,711404,1958\nCPRT,Copart,Industrials,Diversified Support Services,"Dallas, Texas",2018-07-02,900075,1982\nGLW,Corning Inc.,Information Technology,Electronic Components,"Corning, New York",1995-02-27,24741,1851\nCPAY,Corpay,Financials,Transaction & Payment Processing Services,"Atlanta, Georgia",2018-06-20,1175454,2000\nCSGP,CoStar Group,Real Estate,Real Estate Services,"Washington, D.C.",2022-09-19,1057352,1987\nCOST,Costco,Consumer Staples,Consumer Staples Merchandise Retail,"Issaquah, Washington",1993-10-01,909832,1976\nCRH,CRH plc,Materials,Construction Materials,"Dublin, Ireland",2025-12-22,849395,1970\nCRWD,CrowdStrike,Information Technology,Systems Software,"Austin, Texas",2024-06-24,1535527,2011\nCCI,Crown Castle,Real Estate,Telecom Tower REITs,"Houston, Texas",2012-03-14,1051470,1994\nCSX,CSX Corporation,Industrials,Rail Transportation,"Jacksonville, Florida",1957-03-04,277948,1980\nCMI,Cummins,Industrials,Construction Machinery & Heavy Transportation Equipment,"Columbus, Indiana",1965-03-31,26172,1919\nCVS,CVS Health,Health Care,Health Care Services,"Woonsocket, Rhode Island",1957-03-04,64803,1996\nDHR,Danaher Corporation,Health Care,Life Sciences Tools & Services,"Washington, D.C.",1998-11-18,313616,1969\nDRI,Darden Restaurants,Consumer Discretionary,Restaurants,"Orlando, Florida",1995-05-31,940944,1938\nDDOG,Datadog,Information Technology,Application Software,"New York City, New York",2025-07-09,1561550,2010\nDVA,DaVita,Health Care,Health Care Services,"Denver, Colorado",2008-07-31,927066,1979\nDECK,Deckers Brands,Consumer Discretionary,Footwear,"Goleta, California",2024-03-18,910521,1973\nDE,Deere & Company,Industrials,Agricultural & Farm Machinery,"Moline, Illinois",1957-03-04,315189,1837\nDELL,Dell Technologies,Information Technology,"Technology Hardware, Storage & Peripherals","Round Rock, Texas",2024-09-23,1571996,2016\nDAL,Delta Air Lines,Industrials,Passenger Airlines,"Atlanta, Georgia",2013-09-11,27904,1929\nDVN,Devon Energy,Energy,Oil & Gas Exploration & Production,"Oklahoma City, Oklahoma",2000-08-30,1090012,1971\nDXCM,Dexcom,Health Care,Health Care Equipment,"San Diego, California",2020-05-12,1093557,1999\nFANG,Diamondback Energy,Energy,Oil & Gas Exploration & Production,"Midland, Texas",2018-12-03,1539838,2007\nDLR,Digital Realty,Real Estate,Data Center REITs,"Austin, Texas",2016-05-18,1297996,2004\nDG,Dollar General,Consumer Staples,Consumer Staples Merchandise Retail,"Goodlettsville, Tennessee",2012-12-03,29534,1939\nDLTR,Dollar Tree,Consumer Staples,Consumer Staples Merchandise Retail,"Chesapeake, Virginia",2011-12-19,935703,1986\nD,Dominion Energy,Utilities,Multi-Utilities,"Richmond, Virginia",2016-11-30,715957,1983\nDPZ,Domino\'s,Consumer Discretionary,Restaurants,"Ann Arbor, Michigan",2020-05-12,1286681,1960\nDASH,DoorDash,Consumer Discretionary,Specialized Consumer Services,"San Francisco, California",2025-03-24,1792789,2012\nDOV,Dover Corporation,Industrials,Industrial Machinery & Supplies & Components,"Downers Grove, Illinois",1985-10-31,29905,1955\nDOW,Dow Inc.,Materials,Commodity Chemicals,"Midland, Michigan",2019-04-01,1751788,2019 (1897)\nDHI,D. R. Horton,Consumer Discretionary,Homebuilding,"Arlington, Texas",2005-06-22,882184,1978\nDTE,DTE Energy,Utilities,Multi-Utilities,"Detroit, Michigan",1957-03-04,936340,1995\nDUK,Duke Energy,Utilities,Electric Utilities,"Charlotte, North Carolina",1976-06-30,1326160,1904\nDD,DuPont,Industrials,Industrial Conglomerates,"Wilmington, Delaware",2019-06-03,1666700,2017 (1802)\nETN,Eaton Corporation,Industrials,Electrical Components & Equipment,"Dublin, Ireland",1957-03-04,1551182,1911\nEBAY,eBay Inc.,Consumer Discretionary,Broadline Retail,"San Jose, California",2002-07-22,1065088,1995\nECHO,EchoStar,Communication Services,Wireless Telecommunication Services,"Englewood, Colorado",2026-03-23,1415404,2008\nECL,Ecolab,Materials,Specialty Chemicals,"Saint Paul, Minnesota",1989-01-31,31462,1923\nEIX,Edison International,Utilities,Electric Utilities,"Rosemead, California",1957-03-04,827052,1886\nEW,Edwards Lifesciences,Health Care,Health Care Equipment,"Irvine, California",2011-04-01,1099800,1958\nELV,Elevance Health,Health Care,Managed Health Care,"Indianapolis, Indiana",2002-07-25,1156039,2014 (1946)\nEME,Emcor,Industrials,Construction & Engineering,"Norwalk, Connecticut",2025-09-22,105634,1994\nEMR,Emerson Electric,Industrials,Electrical Components & Equipment,"Ferguson, Missouri",1965-03-31,32604,1890\nETR,Entergy,Utilities,Electric Utilities,"New Orleans, Louisiana",1957-03-04,65984,1913\nEOG,EOG Resources,Energy,Oil & Gas Exploration & Production,"Houston, Texas",2000-11-02,821189,1999\nEQT,EQT Corporation,Energy,Oil & Gas Exploration & Production,"Pittsburgh, Pennsylvania",2022-10-03,33213,1888\nEFX,Equifax,Industrials,Research & Consulting Services,"Atlanta, Georgia",1997-06-19,33185,1899\nEQIX,Equinix,Real Estate,Data Center REITs,"Redwood City, California",2015-03-20,1101239,1998\nERIE,Erie Indemnity,Financials,Insurance Brokers,"Erie, Pennsylvania",2024-09-23,922621,1925\nESS,Essex Property Trust,Real Estate,Multi-Family Residential REITs,"San Mateo, California",2014-04-02,920522,1971\nEL,Estée Lauder Companies (The),Consumer Staples,Personal Care Products,"New York City, New York",2006-01-05,1001250,1946\nEG,Everest Group,Financials,Reinsurance,"Hamilton, Bermuda",2017-06-19,1095073,1973\nEVRG,Evergy,Utilities,Electric Utilities,"Kansas City, Missouri",2018-06-05,1711269,1909\nP,Everpure,Information Technology,"Technology Hardware, Storage & Peripherals","Santa Clara, California",2026-09-21,1474432,2009\nES,Eversource Energy,Utilities,Electric Utilities,"Hartford, Connecticut",2009-07-24,72741,1966\nEXC,Exelon,Utilities,Electric Utilities,"Chicago, Illinois",1957-03-04,1109357,2000\nEXE,Expand Energy,Energy,Oil & Gas Exploration & Production,"Oklahoma City, Oklahoma",2025-03-24,895126,1989\nEXPE,Expedia Group,Consumer Discretionary,"Hotels, Resorts & Cruise Lines","Seattle, Washington",2007-10-02,1324424,1996\nEXPD,Expeditors International,Industrials,Air Freight & Logistics,"Seattle, Washington",2007-10-10,746515,1979\nEXR,Extra Space Storage,Real Estate,Self-Storage REITs,"Salt Lake City, Utah",2016-01-19,1289490,1977\nXOM,ExxonMobil,Energy,Integrated Oil & Gas,"Irving, Texas",1957-03-04,2115436,1999\nFFIV,"F5, Inc.",Information Technology,Communications Equipment,"Seattle, Washington",2010-12-20,1048695,1996\nFDS,FactSet,Financials,Financial Exchanges & Data,"Norwalk, Connecticut",2021-12-20,1013237,1978\nFICO,Fair Isaac,Information Technology,Application Software,"Bozeman, Montana",2023-03-20,814547,1956\nFAST,Fastenal,Industrials,Trading Companies & Distributors,"Winona, Minnesota",2008-09-15,815556,1967\nFRT,Federal Realty Investment Trust,Real Estate,Retail REITs,"Rockville, Maryland",2016-02-01,34903,1962\nFDX,FedEx,Industrials,Air Freight & Logistics,"Memphis, Tennessee",1980-12-31,1048911,1971\nFDXF,FedEx Freight,Industrials,Cargo Ground Transportation,"Memphis, Tennessee",2026-06-01,2082247,1982\nFERG,Ferguson Enterprises,Industrials,Trading Companies & Distributors,"Newport News, Virginia",2026-08-05,2011641,1953\nFIS,Fidelity National Information Services,Financials,Transaction & Payment Processing Services,"Jacksonville, Florida",2006-11-10,1136893,1968\nFITB,Fifth Third Bancorp,Financials,Regional Banks,"Cincinnati, Ohio",1996-03-29,35527,1858\nFSLR,First Solar,Information Technology,Semiconductors,"Tempe, Arizona",2022-12-19,1274494,1999\nFE,FirstEnergy,Utilities,Electric Utilities,"Akron, Ohio",1997-11-28,1031296,1997\nFISV,Fiserv,Financials,Transaction & Payment Processing Services,"Brookfield, Wisconsin",2001-04-02,798354,1984\nFLEX,Flex Ltd.,Information Technology,Electronic Manufacturing Services,"Austin, Texas",2026-06-22,866374,1969\nF,Ford Motor Company,Consumer Discretionary,Automobile Manufacturers,"Dearborn, Michigan",1957-03-04,37996,1903\nFTNT,Fortinet,Information Technology,Systems Software,"Sunnyvale, California",2018-10-11,1262039,2000\nFTV,Fortive,Industrials,Industrial Machinery & Supplies & Components,"Everett, Washington",2016-07-01,1659166,2016\nFOXA,Fox Corporation (Class A),Communication Services,Broadcasting,"New York City, New York",2019-03-19,1754301,2019\nFOX,Fox Corporation (Class B),Communication Services,Broadcasting,"New York City, New York",2019-03-19,1754301,2019\nBEN,Franklin Resources,Financials,Asset Management & Custody Banks,"San Mateo, California",1998-04-30,38777,1947\nFCX,Freeport-McMoRan,Materials,Copper,"Phoenix, Arizona",2011-07-01,831259,1912\nGRMN,Garmin,Consumer Discretionary,Consumer Electronics,"Schaffhausen, Switzerland",2012-12-12,1121788,1989\nIT,Gartner,Information Technology,IT Consulting & Other Services,"Stamford, Connecticut",2017-04-05,749251,1979\nGE,GE Aerospace,Industrials,Aerospace & Defense,"Evendale, Ohio",1957-03-04,40545,1892\nGEHC,GE HealthCare,Health Care,Health Care Equipment,"Chicago, Illinois",2023-01-04,1932393,1994\nGEV,GE Vernova,Industrials,Heavy Electrical Equipment,"Cambridge, Massachusetts",2024-04-02,1996810,2024\nGEN,Gen Digital,Information Technology,Systems Software,"Tempe, Arizona",2003-03-25,849399,1982\nGNRC,Generac,Industrials,Heavy Electrical Equipment,"Waukesha, Wisconsin",2021-03-22,1474735,1959\nGD,General Dynamics,Industrials,Aerospace & Defense,"Reston, Virginia",1957-03-04,40533,1899\nGIS,General Mills,Consumer Staples,Packaged Foods & Meats,"Golden Valley, Minnesota",1957-03-04,40704,1856\nGM,General Motors,Consumer Discretionary,Automobile Manufacturers,"Detroit, Michigan",2013-06-06,1467858,1908\nGPC,Genuine Parts Company,Consumer Discretionary,Distributors,"Atlanta, Georgia",1973-12-31,40987,1925\nGILD,Gilead Sciences,Health Care,Biotechnology,"Foster City, California",2004-07-01,882095,1987\nGPN,Global Payments,Financials,Transaction & Payment Processing Services,"Atlanta, Georgia",2016-04-25,1123360,2000\nGL,Globe Life,Financials,Life & Health Insurance,"McKinney, Texas",1989-04-30,320335,1900\nGDDY,GoDaddy,Information Technology,Internet Services & Infrastructure,"Tempe, Arizona",2024-06-24,1609711,1997\nGS,Goldman Sachs,Financials,Investment Banking & Brokerage,"New York City, New York",2002-07-22,886982,1869\nHAL,Halliburton,Energy,Oil & Gas Equipment & Services,"Houston, Texas",1957-03-04,45012,1919\nHIG,Hartford (The),Financials,Property & Casualty Insurance,"Hartford, Connecticut",1957-03-04,874766,1810\nHAS,Hasbro,Consumer Discretionary,Leisure Products,"Pawtucket, Rhode Island",1984-09-30,46080,1923\nHCA,HCA Healthcare,Health Care,Health Care Facilities,"Nashville, Tennessee",2015-01-27,860730,1968\nDOC,Healthpeak Properties,Real Estate,Health Care REITs,"Denver, Colorado",2008-03-31,765880,1985\nHSIC,Henry Schein,Health Care,Health Care Distributors,"Melville, New York",2015-03-17,1000228,1932\nHSY,Hershey Company (The),Consumer Staples,Packaged Foods & Meats,"Hershey, Pennsylvania",1957-03-04,47111,1894\nHPE,Hewlett Packard Enterprise,Information Technology,"Technology Hardware, Storage & Peripherals","Houston, Texas",2015-11-02,1645590,2015\nHLT,Hilton Worldwide,Consumer Discretionary,"Hotels, Resorts & Cruise Lines","Tysons Corner, Virginia",2017-06-19,1585689,1919\nHD,Home Depot (The),Consumer Discretionary,Home Improvement Retail,"Atlanta, Georgia",1988-03-31,354950,1978\nHONA,Honeywell Aerospace,Industrials,Aerospace & Defense,"Phoenix, Arizona",2026-06-29,2089271,1914\nHON,Honeywell Technologies,Industrials,Industrial Conglomerates,"Charlotte, North Carolina",1957-03-04,773840,1906\nHRL,Hormel Foods,Consumer Staples,Packaged Foods & Meats,"Austin, Minnesota",2009-03-04,48465,1891\nHST,Host Hotels & Resorts,Real Estate,Hotel & Resort REITs,"Bethesda, Maryland",2007-03-20,1070750,1993\nHWM,Howmet Aerospace,Industrials,Aerospace & Defense,"Pittsburgh, Pennsylvania",2016-10-21,4281,1888\nHPQ,HP Inc.,Information Technology,"Technology Hardware, Storage & Peripherals","Palo Alto, California",1974-12-31,47217,1939 (2015)\nHUBB,Hubbell Incorporated,Industrials,Industrial Machinery & Supplies & Components,"Shelton, Connecticut",2023-10-18,48898,1888\nHUM,Humana,Health Care,Managed Health Care,"Louisville, Kentucky",2012-12-10,49071,1961\nHBAN,Huntington Bancshares,Financials,Regional Banks,"Columbus, Ohio; Detroit, Michigan",1997-08-28,49196,1866\nHII,Huntington Ingalls Industries,Industrials,Aerospace & Defense,"Newport News, Virginia",2018-01-03,1501585,2011\nIBM,IBM,Information Technology,IT Consulting & Other Services,"Armonk, New York",1957-03-04,51143,1911\nIEX,IDEX Corporation,Industrials,Industrial Machinery & Supplies & Components,"Northbrook, Illinois",2019-08-09,832101,1988\nIDXX,Idexx Laboratories,Health Care,Health Care Equipment,"Westbrook, Maine",2017-01-05,874716,1983\nITW,Illinois Tool Works,Industrials,Industrial Machinery & Supplies & Components,"Glenview, Illinois",1986-02-28,49826,1912\nILMN,"Illumina, Inc.",Health Care,Life Sciences Tools & Services,"San Diego, California",2026-09-21,1110803,1998\nINCY,Incyte,Health Care,Biotechnology,"Wilmington, Delaware",2017-02-28,879169,1991\nIR,Ingersoll Rand,Industrials,Industrial Machinery & Supplies & Components,"Davidson, North Carolina",2020-03-03,1699150,1859\nPODD,Insulet Corporation,Health Care,Health Care Equipment,"Acton, Massachusetts",2023-03-15,1145197,2000\nINTC,Intel,Information Technology,Semiconductors,"Santa Clara, California",1976-12-31,50863,1968\nIBKR,Interactive Brokers,Financials,Investment Banking & Brokerage,"Greenwich, Connecticut",2025-08-28,1381197,1977\nICE,Intercontinental Exchange,Financials,Financial Exchanges & Data,"Atlanta, Georgia",2007-09-26,1571949,2000\nIFF,International Flavors & Fragrances,Materials,Specialty Chemicals,"New York City, New York",1976-03-31,51253,1958 (1889)\nIP,International Paper,Materials,Paper & Plastic Packaging Products & Materials,"Memphis, Tennessee",1957-03-04,51434,1898\nINTU,Intuit,Information Technology,Application Software,"Mountain View, California",2000-12-05,896878,1983\nISRG,Intuitive Surgical,Health Care,Health Care Equipment,"Sunnyvale, California",2008-06-02,1035267,1995\nIVZ,Invesco,Financials,Asset Management & Custody Banks,"Atlanta, Georgia",2008-08-21,914208,1935\nINVH,Invitation Homes,Real Estate,Single-Family Residential REITs,"Dallas, Texas",2022-09-19,1687229,2012\nIQV,IQVIA,Health Care,Life Sciences Tools & Services,"Durham, North Carolina",2017-08-29,1478242,1982\nIRM,Iron Mountain,Real Estate,Other Specialized REITs,"Portsmouth, New Hampshire",2009-01-06,1020569,1951\nJBHT,J.B. Hunt,Industrials,Cargo Ground Transportation,"Lowell, Arkansas",2015-07-01,728535,1961\nJBL,Jabil,Information Technology,Electronic Manufacturing Services,"St. Petersburg, Florida",2023-12-18,898293,1966\nJKHY,Jack Henry & Associates,Financials,Transaction & Payment Processing Services,"Monett, Missouri",2018-11-13,779152,1976\nJ,Jacobs Solutions,Industrials,Construction & Engineering,"Dallas, Texas",2007-10-26,52988,1947\nJNJ,Johnson & Johnson,Health Care,Pharmaceuticals,"New Brunswick, New Jersey",1973-06-30,200406,1886\nJCI,Johnson Controls,Industrials,Building Products,"Cork, Ireland",2010-08-27,833444,1885\nJPM,JPMorgan Chase,Financials,Diversified Banks,"New York City, New York",1975-06-30,19617,2000 (1799 / 1871)\nKVUE,Kenvue,Consumer Staples,Personal Care Products,"Skillman, New Jersey",2023-08-25,1944048,2022 (Johnson & Johnson spinoff)\nKDP,Keurig Dr Pepper,Consumer Staples,Soft Drinks & Non-alcoholic Beverages,"Burlington, Massachusetts",2022-06-21,1418135,1981\nKEY,KeyCorp,Financials,Regional Banks,"Cleveland, Ohio",1994-03-01,91576,1825\nKEYS,Keysight Technologies,Information Technology,Electronic Equipment & Instruments,"Santa Rosa, California",2018-11-06,1601046,2014 (1939)\nKMB,Kimberly-Clark,Consumer Staples,Household Products,"Irving, Texas",1957-03-04,55785,1872\nKIM,Kimco Realty,Real Estate,Retail REITs,"Jericho, New York",2006-04-04,879101,1958\nKMI,Kinder Morgan,Energy,Oil & Gas Storage & Transportation,"Houston, Texas",2012-05-25,1506307,1997\nKKR,KKR & Co.,Financials,Asset Management & Custody Banks,"New York City, New York",2024-06-24,1404912,1976\nKLAC,KLA Corporation,Information Technology,Semiconductor Materials & Equipment,"Milpitas, California",1997-09-30,319201,1975/1977 (1997)\nKHC,Kraft Heinz,Consumer Staples,Packaged Foods & Meats,"Chicago, Illinois; Pittsburgh, Pennsylvania",2015-07-06,1637459,2015 (1869)\nKR,Kroger,Consumer Staples,Food Retail,"Cincinnati, Ohio",1957-03-04,56873,1883\nLHX,L3Harris,Industrials,Aerospace & Defense,"Melbourne, Florida",2008-09-22,202058,"2019 (L3 1997, Harris 1895)"\nLH,Labcorp,Health Care,Health Care Services,"Burlington, North Carolina",2004-11-01,920148,1978\nLRCX,Lam Research,Information Technology,Semiconductor Materials & Equipment,"Fremont, California",2012-06-29,707549,1980\nLVS,Las Vegas Sands,Consumer Discretionary,Casinos & Gaming,"Las Vegas, Nevada",2019-10-03,1300514,1988\nLDOS,Leidos,Industrials,Diversified Support Services,"Reston, Virginia",2019-08-09,1336920,1969\nLEN,Lennar,Consumer Discretionary,Homebuilding,"Miami, Florida",2005-10-04,920760,1954\nLII,Lennox International,Industrials,Building Products,"Richardson, Texas",2024-12-23,1069202,1895\nLLY,Lilly (Eli),Health Care,Pharmaceuticals,"Indianapolis, Indiana",1970-12-31,59478,1876\nLIN,Linde plc,Materials,Industrial Gases,"Guildford, United Kingdom",1992-07-01,1707925,1879\nLYV,Live Nation Entertainment,Communication Services,Movies & Entertainment,"Beverly Hills, California",2019-12-23,1335258,2010\nLMT,Lockheed Martin,Industrials,Aerospace & Defense,"Bethesda, Maryland",1957-03-04,936468,1995\nL,Loews Corporation,Financials,Multi-line Insurance,"New York City, New York",1995-05-31,60086,1959\nLOW,Lowe\'s,Consumer Discretionary,Home Improvement Retail,"Mooresville, North Carolina",1984-02-29,60667,1904/1946/1959\nLULU,Lululemon Athletica,Consumer Discretionary,"Apparel, Accessories & Luxury Goods","Vancouver, Canada",2023-10-18,1397187,1998\nLITE,Lumentum,Information Technology,Communications Equipment,"San Jose, California",2026-03-23,1633978,2015\nLYB,LyondellBasell,Materials,Specialty Chemicals,"Rotterdam, Netherlands",2012-09-05,1489393,2007\nMTB,M&T Bank,Financials,Regional Banks,"Buffalo, New York",2004-02-23,36270,1856\nMPC,Marathon Petroleum,Energy,Oil & Gas Refining & Marketing,"Findlay, Ohio",2011-07-01,1510295,2009 (1887)\nMAR,Marriott International,Consumer Discretionary,"Hotels, Resorts & Cruise Lines","Bethesda, Maryland",1998-05-29,1048286,1927\nMRSH,Marsh McLennan,Financials,Insurance Brokers,"New York City, New York",1987-08-31,62709,1905\nMLM,Martin Marietta Materials,Materials,Construction Materials,"Raleigh, North Carolina",2014-07-02,916076,1993\nMRVL,Marvell Technology,Information Technology,Semiconductors,"Santa Clara, California",2026-06-22,1835632,1995\nMAS,Masco,Industrials,Building Products,"Livonia, Michigan",1981-06-30,62996,1929\nMA,Mastercard,Financials,Transaction & Payment Processing Services,"Harrison, New York",2008-07-18,1141391,1966\nMKC,McCormick & Company,Consumer Staples,Packaged Foods & Meats,"Hunt Valley, Maryland",2003-03-20,63754,1889\nMCD,McDonald\'s,Consumer Discretionary,Restaurants,"Chicago, Illinois",1970-06-30,63908,1940\nMCK,McKesson,Health Care,Health Care Distributors,"Irving, Texas",1999-01-13,927653,1833\nMDT,Medtronic,Health Care,Health Care Equipment,"Galway, Ireland",1986-10-31,1613103,1949\nMRK,Merck & Co.,Health Care,Pharmaceuticals,"Kenilworth, New Jersey",1957-03-04,310158,1891\nMETA,Meta Platforms,Communication Services,Interactive Media & Services,"Menlo Park, California",2013-12-23,1326801,2004\nMET,MetLife,Financials,Life & Health Insurance,"New York City, New York",2000-12-11,1099219,1868\nMTD,Mettler Toledo,Health Care,Life Sciences Tools & Services,"Columbus, Ohio",2016-09-06,1037646,1945\nMGM,MGM Resorts,Consumer Discretionary,Casinos & Gaming,"Paradise, Nevada",2017-07-26,789570,1986\nMCHP,Microchip Technology,Information Technology,Semiconductors,"Chandler, Arizona",2007-09-07,827054,1989\nMU,Micron Technology,Information Technology,Semiconductors,"Boise, Idaho",1994-09-27,723125,1978\nMSFT,Microsoft,Information Technology,Systems Software,"Redmond, Washington",1994-06-01,789019,1975\nMAA,Mid-America Apartment Communities,Real Estate,Multi-Family Residential REITs,"Memphis, Tennessee",2016-12-02,912595,1977\nMRNA,Moderna,Health Care,Biotechnology,"Cambridge, Massachusetts",2021-07-21,1682852,2010\nMDLZ,Mondelez International,Consumer Staples,Packaged Foods & Meats,"Chicago, Illinois",2012-10-02,1103982,2012\nMPWR,Monolithic Power Systems,Information Technology,Semiconductors,"Kirkland, Washington",2021-02-12,1280452,1997\nMNST,Monster Beverage,Consumer Staples,Soft Drinks & Non-alcoholic Beverages,"Corona, California",2012-06-28,865752,2012 (1935)\nMCO,Moody\'s Corporation,Financials,Financial Exchanges & Data,"New York City, New York",1998-07-01,1059556,1909\nMS,Morgan Stanley,Financials,Investment Banking & Brokerage,"New York City, New York",1993-07-29,895421,1935\nMOS,Mosaic Company (The),Materials,Fertilizers & Agricultural Chemicals,"Tampa, Florida",2011-09-26,1285785,2004 (1865 / 1909)\nMSI,Motorola Solutions,Information Technology,Communications Equipment,"Chicago, Illinois",1957-03-04,68505,1928 (2011)\nMSCI,MSCI,Financials,Financial Exchanges & Data,"New York City, New York",2018-04-04,1408198,1969\nNDAQ,"Nasdaq, Inc.",Financials,Financial Exchanges & Data,"New York City, New York",2008-10-22,1120193,1971\nNTAP,NetApp,Information Technology,"Technology Hardware, Storage & Peripherals","San Jose, California",1999-06-25,1002047,1992\nNFLX,Netflix,Communication Services,Movies & Entertainment,"Los Gatos, California",2010-12-20,1065280,1997\nNEM,Newmont,Materials,Gold,"Denver, Colorado",1969-06-30,1164727,1921\nNWSA,News Corp (Class A),Communication Services,Publishing,"New York City, New York",2013-08-01,1564708,2013 (News Corporation 1980)\nNWS,News Corp (Class B),Communication Services,Publishing,"New York City, New York",2015-09-18,1564708,2013 (News Corporation 1980)\nNEE,NextEra Energy,Utilities,Multi-Utilities,"Juno Beach, Florida",1976-06-30,753308,1984 (1925)\nNKE,"Nike, Inc.",Consumer Discretionary,"Apparel, Accessories & Luxury Goods","Washington County, Oregon",1988-11-30,320187,1964\nNI,NiSource,Utilities,Multi-Utilities,"Merrillville, Indiana",2000-11-02,1111711,1912\nNDSN,Nordson Corporation,Industrials,Industrial Machinery & Supplies & Components,"Westlake, Ohio",2022-02-15,72331,1935\nNSC,Norfolk Southern,Industrials,Rail Transportation,"Atlanta, Georgia",1957-03-04,702165,1881/1894 (1980)\nNTRS,Northern Trust,Financials,Asset Management & Custody Banks,"Chicago, Illinois",1998-01-30,73124,1889\nNOC,Northrop Grumman,Industrials,Aerospace & Defense,"West Falls Church, Virginia[2]",1957-03-04,1133421,"1994 (Northrop 1939, Grumman 1930)"\nNCLH,Norwegian Cruise Line Holdings,Consumer Discretionary,"Hotels, Resorts & Cruise Lines","Miami-Dade County, Florida[3]",2017-10-13,1513761,2011 (1966)\nNRG,NRG Energy,Utilities,Independent Power Producers & Energy Traders,"Houston, Texas",2010-01-29,1013871,1992\nNUE,Nucor,Materials,Steel,"Charlotte, North Carolina",1985-04-30,73309,1940\nNVDA,Nvidia,Information Technology,Semiconductors,"Santa Clara, California",2001-11-30,1045810,1993\nNVR,"NVR, Inc.",Consumer Discretionary,Homebuilding,"Reston, Virginia",2019-09-26,906163,1980\nNXPI,NXP Semiconductors,Information Technology,Semiconductors,"Eindhoven, Netherlands",2021-03-22,1413447,1953\nORLY,O\'Reilly Automotive,Consumer Discretionary,Automotive Retail,"Springfield, Missouri",2009-03-27,898173,1957\nOXY,Occidental Petroleum,Energy,Oil & Gas Exploration & Production,"Houston, Texas",1957-03-04,797468,1920\nODFL,Old Dominion,Industrials,Cargo Ground Transportation,"Thomasville, North Carolina",2019-12-09,878927,1934\nOMC,Omnicom Group,Communication Services,Advertising,"New York City, New York",1997-12-31,29989,1986\nON,ON Semiconductor,Information Technology,Semiconductors,"Phoenix, Arizona",2022-06-21,1097864,1999\nOKE,Oneok,Energy,Oil & Gas Storage & Transportation,"Tulsa, Oklahoma",2010-03-15,1039684,1906\nORCL,Oracle Corporation,Information Technology,Application Software,"Austin, Texas",1989-08-31,1341439,1977\nOTIS,Otis Worldwide,Industrials,Industrial Machinery & Supplies & Components,"Farmington, Connecticut",2020-04-03,1781335,"2020 (1853, United Technologies spinoff)"\nPCAR,Paccar,Industrials,Construction Machinery & Heavy Transportation Equipment,"Bellevue, Washington",1980-12-31,75362,1905\nPKG,Packaging Corporation of America,Materials,Paper & Plastic Packaging Products & Materials,"Lake Forest, Illinois",2017-07-26,75677,1959\nPLTR,Palantir Technologies,Information Technology,Application Software,"Aventura, Florida",2024-09-23,1321655,2003\nPANW,Palo Alto Networks,Information Technology,Systems Software,"Santa Clara, California",2023-06-20,1327567,2005\nPH,Parker Hannifin,Industrials,Industrial Machinery & Supplies & Components,"Cleveland, Ohio",1985-11-30,76334,1917\nPAYX,Paychex,Industrials,Human Resource & Employment Services,"Penfield, New York",1998-10-01,723531,1971\nPYPL,PayPal,Financials,Transaction & Payment Processing Services,"San Jose, California",2015-07-20,1633917,1998\nPNR,Pentair,Industrials,Industrial Machinery & Supplies & Components,"Worsley, United Kingdom",2012-10-01,77360,1966\nPEP,PepsiCo,Consumer Staples,Soft Drinks & Non-alcoholic Beverages,"Purchase, New York",1957-03-04,77476,1898\nPFE,Pfizer,Health Care,Pharmaceuticals,"New York City, New York",1957-03-04,78003,1849\nPCG,PG&E Corporation,Utilities,Multi-Utilities,"Oakland, California",2022-10-03,1004980,1905\nPM,Philip Morris International,Consumer Staples,Tobacco,"New York City, New York",2008-03-31,1413329,2008 (1847)\nPSX,Phillips 66,Energy,Oil & Gas Refining & Marketing,"Houston, Texas",2012-05-01,1534701,2012 (1917)\nPNW,Pinnacle West Capital,Utilities,Multi-Utilities,"Phoenix, Arizona",1999-10-04,764622,1985\nPNC,PNC Financial Services,Financials,Diversified Banks,"Pittsburgh, Pennsylvania",1988-04-30,713676,1845\nPPG,PPG Industries,Materials,Specialty Chemicals,"Pittsburgh, Pennsylvania",1957-03-04,79879,1883\nPPL,PPL Corporation,Utilities,Electric Utilities,"Allentown, Pennsylvania",2001-10-01,922224,1920\nPFG,Principal Financial Group,Financials,Life & Health Insurance,"Des Moines, Iowa",2002-07-22,1126328,1879\nPG,Procter & Gamble,Consumer Staples,Personal Care Products,"Cincinnati, Ohio",1957-03-04,80424,1837\nPGR,Progressive Corporation,Financials,Property & Casualty Insurance,"Mayfield Village, Ohio",1997-08-04,80661,1937\nPLD,Prologis,Real Estate,Industrial REITs,"San Francisco, California",2003-07-17,1045609,1983\nPRU,Prudential Financial,Financials,Life & Health Insurance,"Newark, New Jersey",2002-07-22,1137774,1875\nPTC,PTC Inc.,Information Technology,Application Software,"Boston, Massachusetts",2021-04-20,857005,1985\nPEG,Public Service Enterprise Group,Utilities,Electric Utilities,"Newark, New Jersey",1957-03-04,788784,1903\nPSA,Public Storage,Real Estate,Self-Storage REITs,"Glendale, California",2005-08-19,1393311,1972\nPHM,PulteGroup,Consumer Discretionary,Homebuilding,"Atlanta, Georgia",1984-04-30,822416,1956\nQ,Qnity Electronics,Information Technology,Semiconductor Materials & Equipment,"Wilmington, Delaware",2025-11-03,2058873,2025\nQCOM,Qualcomm,Information Technology,Semiconductors,"San Diego, California",1999-07-22,804328,1985\nPWR,Quanta Services,Industrials,Construction & Engineering,"Houston, Texas",2009-07-01,1050915,1997\nDGX,Quest Diagnostics,Health Care,Health Care Services,"Secaucus, New Jersey",2002-12-12,1022079,1967\nRL,Ralph Lauren Corporation,Consumer Discretionary,"Apparel, Accessories & Luxury Goods","New York City, New York",2007-02-02,1037038,1967\nRJF,Raymond James Financial,Financials,Investment Banking & Brokerage,"St. Petersburg, Florida",2017-03-20,720005,1962\nRDDT,Reddit,Communication Services,Interactive Media & Services,"San Francisco, California",2026-08-18,1713445,2005\nO,Realty Income,Real Estate,Retail REITs,"San Diego, California",2015-04-07,726728,1969\nREG,Regency Centers,Real Estate,Retail REITs,"Jacksonville, Florida",2017-03-02,910606,1963\nREGN,Regeneron Pharmaceuticals,Health Care,Biotechnology,"Tarrytown, New York",2013-05-01,872589,1988\nRF,Regions Financial Corporation,Financials,Regional Banks,"Birmingham, Alabama",1998-08-28,1281761,1971\nRSG,Republic Services,Industrials,Environmental & Facilities Services,"Phoenix, Arizona",2008-12-05,1060391,1998 (1981)\nRMD,ResMed,Health Care,Health Care Equipment,"San Diego, California",2017-07-26,943819,1989\nRVTY,Revvity,Health Care,Health Care Equipment,"Waltham, Massachusetts",1985-05-31,31791,1937\nHOOD,Robinhood Markets,Financials,Investment Banking & Brokerage,"Menlo Park, California",2025-09-22,1783879,2013\nROK,Rockwell Automation,Industrials,Electrical Components & Equipment,"Milwaukee, Wisconsin",2000-03-12,1024478,1903\nROL,"Rollins, Inc.",Industrials,Environmental & Facilities Services,"Atlanta, Georgia",2018-10-01,84839,1948\nROP,Roper Technologies,Information Technology,Electronic Equipment & Instruments,"Sarasota, Florida",2009-12-23,882835,1981\nROST,Ross Stores,Consumer Discretionary,Apparel Retail,"Dublin, California",2009-12-21,745732,1982\nRCL,Royal Caribbean Group,Consumer Discretionary,"Hotels, Resorts & Cruise Lines","Miami, Florida",2014-12-05,884887,1997\nRTX,RTX Corporation,Industrials,Aerospace & Defense,"Waltham, Massachusetts",1957-03-04,101829,1922\nSPGI,S&P Global,Financials,Financial Exchanges & Data,"New York City, New York",1957-03-04,64040,1917\nCRM,Salesforce,Information Technology,Application Software,"San Francisco, California",2008-09-15,1108524,1999\nSNDK,Sandisk,Information Technology,"Technology Hardware, Storage & Peripherals","Milpitas, California",2025-11-28,2023554,1988\nSBAC,SBA Communications,Real Estate,Telecom Tower REITs,"Boca Raton, Florida",2017-09-01,1034054,1989\nSLB,Schlumberger,Energy,Oil & Gas Equipment & Services,"Houston, Texas",1957-03-04,87347,1926\nSTX,Seagate Technology,Information Technology,"Technology Hardware, Storage & Peripherals","Dublin, Ireland",2012-07-02,1137789,1979\nSRE,Sempra,Utilities,Multi-Utilities,"San Diego, California",2017-03-17,1032208,1998\nNOW,ServiceNow,Information Technology,Systems Software,"Santa Clara, California",2019-11-21,1373715,2003\nSHW,Sherwin-Williams,Materials,Specialty Chemicals,"Cleveland, Ohio",1964-06-30,89800,1866\nSPG,Simon Property Group,Real Estate,Retail REITs,"Indianapolis, Indiana",2002-06-26,1063761,2003\nSKYD,Skydance,Communication Services,Movies & Entertainment,"New York City, New York",1994-09-30,2041610,2025 (Paramount Pictures 1912)\nSWKS,Skyworks Solutions,Information Technology,Semiconductors,"Irvine, California",2015-03-12,4127,2002\nSJM,J.M. Smucker Company (The),Consumer Staples,Packaged Foods & Meats,"Orrville, Ohio",2008-11-06,91419,1897\nSW,Smurfit Westrock,Materials,Paper & Plastic Packaging Products & Materials,"Dublin, Ireland",2024-07-08,2005951,1934\nSNA,Snap-on,Industrials,Industrial Machinery & Supplies & Components,"Kenosha, Wisconsin",1982-09-30,91440,1920\nSOLV,Solventum,Health Care,Health Care Technology,"Saint Paul, Minnesota",2024-04-01,1964738,2023\nSO,Southern Company,Utilities,Electric Utilities,"Atlanta, Georgia",1957-03-04,92122,1945\nLUV,Southwest Airlines,Industrials,Passenger Airlines,"Dallas, Texas",1994-07-01,92380,1967\nSWK,Stanley Black & Decker,Industrials,Industrial Machinery & Supplies & Components,"New Britain, Connecticut",1982-09-30,93556,1843\nSBUX,Starbucks,Consumer Discretionary,Restaurants,"Seattle, Washington",2000-06-07,829224,1971\nSTT,State Street Corporation,Financials,Asset Management & Custody Banks,"Boston, Massachusetts",2003-03-14,93751,1792\nSTLD,Steel Dynamics,Materials,Steel,"Fort Wayne, Indiana",2022-12-22,1022671,1993\nSTE,Steris,Health Care,Health Care Equipment,"Dublin, Ireland",2019-12-23,1757898,1985\nSYK,Stryker Corporation,Health Care,Health Care Equipment,"Kalamazoo, Michigan",2000-12-12,310764,1941\nSMCI,Supermicro,Information Technology,"Technology Hardware, Storage & Peripherals","San Jose, California",2024-03-18,1375365,1993\nSYF,Synchrony Financial,Financials,Consumer Finance,"Stamford, Connecticut",2015-11-18,1601712,2003\nSNPS,Synopsys,Information Technology,Application Software,"Sunnyvale, California",2017-03-16,883241,1986\nSYY,Sysco,Consumer Staples,Food Distributors,"Houston, Texas",1986-12-31,96021,1969\nTMUS,T-Mobile US,Communication Services,Wireless Telecommunication Services,"Bellevue, Washington",2019-07-15,1283699,1994\nTROW,T. Rowe Price,Financials,Asset Management & Custody Banks,"Baltimore, Maryland",2019-07-29,1113169,1937\nTTWO,Take-Two Interactive,Communication Services,Interactive Home Entertainment,"New York City, New York",2018-03-19,946581,1993\nTPR,"Tapestry, Inc.",Consumer Discretionary,"Apparel, Accessories & Luxury Goods","New York City, New York",2004-09-01,1116132,2017\nTRGP,Targa Resources,Energy,Oil & Gas Storage & Transportation,"Houston, Texas",2022-10-12,1389170,2005\nTGT,Target Corporation,Consumer Staples,Consumer Staples Merchandise Retail,"Minneapolis, Minnesota",1976-12-31,27419,1902\nTEL,TE Connectivity,Information Technology,Electronic Manufacturing Services,"Galway, Ireland",2011-10-17,1385157,2007\nTDY,Teledyne Technologies,Information Technology,Electronic Equipment & Instruments,"Thousand Oaks, California",2020-06-22,1094285,1960\nTER,Teradyne,Information Technology,Semiconductor Materials & Equipment,"North Reading, Massachusetts",2020-09-21,97210,1960\nTSLA,"Tesla, Inc.",Consumer Discretionary,Automobile Manufacturers,"Austin, Texas",2020-12-21,1318605,2003\nTXN,Texas Instruments,Information Technology,Semiconductors,"Dallas, Texas",2001-03-12,97476,1930\nTPL,Texas Pacific Land Corporation,Energy,Oil & Gas Exploration & Production,"Dallas, Texas",2024-11-26,1811074,1888\nTXT,Textron,Industrials,Aerospace & Defense,"Providence, Rhode Island",1978-12-31,217346,1923\nTMO,Thermo Fisher Scientific,Health Care,Life Sciences Tools & Services,"Waltham, Massachusetts",2004-08-03,97745,2006 (1902)\nTJX,TJX Companies,Consumer Discretionary,Apparel Retail,"Framingham, Massachusetts",1985-09-30,109198,1987\nTKO,TKO Group Holdings,Communication Services,Movies & Entertainment,"New York City, New York",2025-03-24,1973266,2023\nTSCO,Tractor Supply,Consumer Discretionary,Other Specialty Retail,"Brentwood, Tennessee",2014-01-24,916365,1938\nTT,Trane Technologies,Industrials,Building Products,"Dublin, Ireland",2010-11-17,1466258,1871\nTDG,TransDigm Group,Industrials,Aerospace & Defense,"Cleveland, Ohio",2016-06-03,1260221,1993\nTRV,Travelers Companies (The),Financials,Property & Casualty Insurance,"New York City, New York",2002-08-21,86312,1853\nTRMB,Trimble Inc.,Information Technology,Application Software,"Westminster, Colorado",2021-01-21,864749,1978\nTFC,Truist Financial,Financials,Diversified Banks,"Charlotte, North Carolina",1997-12-04,92230,1872\nTWLO,Twilio,Information Technology,Systems Software,"San Francisco, California",2026-10-06,1447669,2008\nTYL,Tyler Technologies,Information Technology,Application Software,"Plano, Texas",2020-06-22,860731,1966\nTSN,Tyson Foods,Consumer Staples,Packaged Foods & Meats,"Springdale, Arkansas",2005-08-10,100493,1935\nUSB,U.S. Bancorp,Financials,Diversified Banks,"Minneapolis, Minnesota",1999-11-01,36104,1968\nUBER,Uber,Industrials,Passenger Ground Transportation,"San Francisco, California",2023-12-18,1543151,2009\nUDR,"UDR, Inc.",Real Estate,Multi-Family Residential REITs,"Highlands Ranch, Colorado",2016-03-07,74208,1972\nULTA,Ulta Beauty,Consumer Discretionary,Other Specialty Retail,"Bolingbrook, Illinois",2016-04-18,1403568,1990\nUNP,Union Pacific Corporation,Industrials,Rail Transportation,"Omaha, Nebraska",1957-03-04,100885,1862\nUAL,United Airlines Holdings,Industrials,Passenger Airlines,"Chicago, Illinois",2015-09-03,100517,1967\nUPS,United Parcel Service,Industrials,Air Freight & Logistics,"Sandy Springs, Georgia",2002-07-22,1090727,1907\nURI,United Rentals,Industrials,Trading Companies & Distributors,"Stamford, Connecticut",2014-09-20,1067701,1997\nUNH,UnitedHealth Group,Health Care,Managed Health Care,"Eden Prairie, Minnesota",1994-07-01,731766,1977\nUHS,Universal Health Services,Health Care,Health Care Facilities,"King of Prussia, Pennsylvania",2014-09-20,352915,1979\nVLO,Valero Energy,Energy,Oil & Gas Refining & Marketing,"San Antonio, Texas",2002-12-20,1035002,1980\nVEEV,Veeva Systems,Health Care,Health Care Technology,"Pleasanton, California",2026-05-07,1393052,2007\nVTR,Ventas,Real Estate,Health Care REITs,"Chicago, Illinois",2009-03-04,740260,1998\nVLTO,Veralto,Industrials,Environmental & Facilities Services,"Waltham, Massachusetts",2023-10-02,1967680,2023\nVRSN,Verisign,Information Technology,Internet Services & Infrastructure,"Reston, Virginia",2006-02-01,1014473,1995\nVRSK,Verisk Analytics,Industrials,Research & Consulting Services,"Jersey City, New Jersey",2015-10-08,1442145,1971\nVZ,Verizon,Communication Services,Integrated Telecommunication Services,"New York City, New York",1983-11-30,732712,1983 (1877)\nVRTX,Vertex Pharmaceuticals,Health Care,Biotechnology,"Boston, Massachusetts",2013-09-23,875320,1989\nVRT,Vertiv,Industrials,Electrical Components & Equipment,"Westerville, Ohio",2026-03-23,1674101,2016\nVTRS,Viatris,Health Care,Pharmaceuticals,"Pittsburgh, Pennsylvania",2004-04-23,1792044,1961\nVICI,Vici Properties,Real Estate,Hotel & Resort REITs,"New York City, New York",2022-06-08,1705696,2017\nV,Visa Inc.,Financials,Transaction & Payment Processing Services,"San Francisco, California",2009-12-21,1403161,1958\nVST,Vistra Corp.,Utilities,Electric Utilities,"Irving, Texas",2024-05-08,1692819,2016\nVMRK,Vivmark Residential,Real Estate,Multi-Family Residential REITs,"Chicago, Illinois",2001-12-03,906107,1969\nVMC,Vulcan Materials Company,Materials,Construction Materials,"Birmingham, Alabama",1999-06-30,1396009,1909\nVYLR,Vylor,Consumer Staples,Agricultural Products & Services,"Johnston, Iowa",2026-10-01,2128626,2026\nWRB,W. R. Berkley Corporation,Financials,Property & Casualty Insurance,"Greenwich, Connecticut",2019-12-05,11544,1967\nGWW,W. W. Grainger,Industrials,Industrial Machinery & Supplies & Components,"Lake Forest, Illinois",1981-06-30,277135,1927\nWAB,Wabtec,Industrials,Construction Machinery & Heavy Transportation Equipment,"Pittsburgh, Pennsylvania",2019-02-27,943452,1999 (1869)\nWMT,Walmart,Consumer Staples,Consumer Staples Merchandise Retail,"Bentonville, Arkansas",1982-08-31,104169,1962\nDIS,Walt Disney Company (The),Communication Services,Movies & Entertainment,"Burbank, California",1976-06-30,1744489,1923\nWM,Waste Management,Industrials,Environmental & Facilities Services,"Houston, Texas",1998-08-31,823768,1968\nWAT,Waters Corporation,Health Care,Life Sciences Tools & Services,"Milford, Massachusetts",2002-01-02,1000697,1958\nWEC,WEC Energy Group,Utilities,Electric Utilities,"Milwaukee, Wisconsin",2008-10-31,783325,1896\nWFC,Wells Fargo,Financials,Diversified Banks,"San Francisco, California",1976-06-30,72971,1852\nWELL,Welltower,Real Estate,Health Care REITs,"Toledo, Ohio",2009-01-30,766704,1970\nWST,West Pharmaceutical Services,Health Care,Health Care Supplies,"Exton, Pennsylvania",2020-05-22,105770,1923\nWDC,Western Digital,Information Technology,"Technology Hardware, Storage & Peripherals","San Jose, California",2009-07-01,106040,1970\nWY,Weyerhaeuser,Real Estate,Timber REITs,"Seattle, Washington",1979-10-01,106535,1900\nWSM,"Williams-Sonoma, Inc.",Consumer Discretionary,Homefurnishing Retail,"San Francisco, California",2025-03-24,719955,1956\nWMB,Williams Companies,Energy,Oil & Gas Storage & Transportation,"Tulsa, Oklahoma",1975-03-31,107263,1908\nWTW,Willis Towers Watson,Financials,Insurance Brokers,"London, United Kingdom",2016-01-05,1140536,2016\nWDAY,"Workday, Inc.",Information Technology,Application Software,"Pleasanton, California",2024-12-23,1327811,2005\nWYNN,Wynn Resorts,Consumer Discretionary,Casinos & Gaming,"Paradise, Nevada",2008-11-14,1174922,2002\nXEL,Xcel Energy,Utilities,Multi-Utilities,"Minneapolis, Minnesota",1957-03-04,72903,1909\nXYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011\nYUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997\nZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969\nZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927\nZTS,Zoetis,Health Care,Pharmaceuticals,"Parsippany, New Jersey",2013-06-21,1555280,1952\n'
with open(base / 'sp500.csv', 'w', encoding='utf-8', newline='') as f:
    f.write(SP500_SNAPSHOT)
print('快照中的证券数量：', len(list(csv.DictReader(io.StringIO(SP500_SNAPSHOT)))))


快照中的证券数量： 503


## 合并新闻并精确去重

In [7]:
import csv, sqlite3, json, time
from collections import Counter
from pathlib import Path

base=Path('work/fnspid')
csv.field_size_limit(100_000_000)
con=sqlite3.connect(base/'news_audit.sqlite')
con.execute('PRAGMA journal_mode=WAL')
con.execute('CREATE TABLE IF NOT EXISTS news (ticker TEXT, date TEXT, title TEXT, url TEXT, PRIMARY KEY(ticker,date,title,url)) WITHOUT ROWID')
symbols={r['Symbol'].replace('.','-') for r in csv.DictReader(open(base/'sp500.csv'))}
summary={}
for filename in ['All_external.csv','nasdaq_exteral_data.csv']:
    counts=Counter(); years=Counter(); batch=[]; start=time.time()
    with open(base/filename,newline='',encoding='utf-8',errors='replace') as f:
        for r in csv.DictReader(f):
            counts['raw']+=1
            d=r.get('Date','').strip(); s=r.get('Stock_symbol','').strip().upper().replace('.','-'); title=r.get('Article_title','').strip()
            if len(d)>=10: years[d[:4]]+=1
            if s not in symbols or not title or not '2018-01-01'<=d[:10]<='2023-12-31': continue
            batch.append((s,d,title,r.get('Url','').strip())); counts['eligible_before_dedup']+=1
            if len(batch)>=10000:
                con.executemany('INSERT OR IGNORE INTO news VALUES (?,?,?,?)',batch); con.commit(); batch=[]
            if counts['raw']%200000==0: print(filename,counts['raw'], 'elapsed',round(time.time()-start),flush=True)
    con.executemany('INSERT OR IGNORE INTO news VALUES (?,?,?,?)',batch); con.commit()
    summary[filename]={'counts':dict(counts),'years':dict(sorted(years.items()))}
    print(filename,summary[filename],flush=True)
json.dump(summary,open(base/'news_audit.json','w'),indent=2)
print('unique eligible',con.execute('SELECT COUNT(*) FROM news').fetchone(),flush=True)


All_external.csv 600000 elapsed 1
All_external.csv 800000 elapsed 2
All_external.csv 3400000 elapsed 10
All_external.csv 3600000 elapsed 11
All_external.csv {'counts': {'raw': 13057514, 'eligible_before_dedup': 264604}, 'years': {'1914': 5, '1969': 1, '1999': 3081, '2000': 16176, '2001': 21974, '2002': 22179, '2003': 21557, '2004': 24386, '2005': 30718, '2006': 36057, '2007': 464439, '2008': 1056009, '2009': 960491, '2010': 1200802, '2011': 1571048, '2012': 1569853, '2013': 1192744, '2014': 1166279, '2015': 1364956, '2016': 686338, '2017': 344705, '2018': 523476, '2019': 578683, '2020': 201557}}
nasdaq_exteral_data.csv 3800000 elapsed 55
nasdaq_exteral_data.csv 4800000 elapsed 61
nasdaq_exteral_data.csv {'counts': {'raw': 15549299, 'eligible_before_dedup': 766098}, 'years': {'1914': 5, '1969': 1, '1999': 3081, '2000': 16176, '2001': 21974, '2002': 22179, '2003': 21557, '2004': 24386, '2005': 30718, '2006': 36057, '2007': 464439, '2008': 1056009, '2009': 961342, '2010': 1211112, '2011':

## 审计价格覆盖

In [8]:
z = zipfile.ZipFile(base / 'full_history.zip')
audit = {}
for symbol in sorted(symbols):
    try:
        stream = z.open(f'full_history/{symbol}.csv')
    except KeyError:
        continue
    with stream:
        rows = list(csv.DictReader(io.TextIOWrapper(stream, encoding='utf-8-sig')))
    dates = [r['date'] for r in rows if '2018-01-01' <= r['date'] <= '2023-12-31']
    counts = Counter(d[:4] for d in dates)
    audit[symbol] = {'counts': dict(counts), 'min': min(dates) if dates else None, 'max': max(dates) if dates else None}
with open(base / 'price_audit.json', 'w') as f:
    json.dump(audit, f, indent=2)
print('有价格文件的候选证券数量：', len(audit))
print('主要年度交易日数量：', dict(Counter(tuple(sorted(a['counts'].items())) for a in audit.values()).most_common(1)[0][0]))


有价格文件的候选证券数量： 437
主要年度交易日数量： {'2018': 251, '2019': 252, '2020': 253, '2021': 252, '2022': 251, '2023': 249}


## 选样、日期匹配与标签

In [6]:
import csv,sqlite3,json,zipfile,io,math,datetime,bisect
from collections import Counter
from pathlib import Path
base=Path('work/fnspid'); out=Path('outputs'); out.mkdir(exist_ok=True)
db=sqlite3.connect(base/'news_audit.sqlite')
audit=json.load(open(base/'price_audit.json'))
ref={r['Symbol'].replace('.','-'):r for r in csv.DictReader(open(base/'sp500.csv'))}
expected=dict(Counter(tuple(sorted(a['counts'].items())) for a in audit.values()).most_common(1)[0][0])
rank=db.execute("SELECT ticker,COUNT(*) FROM news WHERE date>='2018' AND date<'2022' GROUP BY ticker ORDER BY COUNT(*) DESC,ticker").fetchall()
selected=[]; ciks=set()
for s,n in rank:
 a=audit.get(s)
 if not a or a['counts']!=expected: continue
 cik=ref[s]['CIK']
 if cik in ciks: continue
 selected.append((s,n)); ciks.add(cik)
 if len(selected)==50: break
assert len(selected)==50,len(selected)
z=zipfile.ZipFile(base/'full_history.zip'); prices={}; days={}
for s,n in selected:
 vals={}
 for r in csv.DictReader(io.TextIOWrapper(z.open('full_history/'+s+'.csv'),encoding='utf-8-sig')):
  d=r['date']
  if '2018-01-01'<=d<='2023-12-31':
   v=float(r['adj close']); assert math.isfinite(v) and v>0
   vals[d]=v
 prices[s]=vals; days[s]=sorted(vals)
news=Counter(); dropped=Counter(); source=Counter()
# Date-only conservative rule: assign news to first trading day strictly after publication date.
for s,n in selected:
 ds=days[s]
 for date,title,url in db.execute('SELECT date,title,url FROM news WHERE ticker=?',(s,)):
  try: d=datetime.date.fromisoformat(date[:10]).isoformat()
  except ValueError: dropped['invalid_date']+=1; continue
  source['selected_news']+=1
  i=bisect.bisect_right(ds,d)
  if i>=len(ds)-1: dropped['no_feature_day_or_next_close']+=1; continue
  news[s,ds[i]]+=1
stats={k:{'companies':set(),'news':0,'company_days':0,'up':0,'down_or_flat':0,'no_news_days':0} for k in ['Training','Validation','Test']}
with open(out/'company_day_labels.csv','w',newline='') as f:
 w=csv.writer(f); w.writerow(['Ticker','Trading_Date','Split','News_Count','HasNews','Adjusted_Close','Next_Adjusted_Close','Label'])
 for s,n in selected:
  ds=days[s]
  for i,d in enumerate(ds[:-1]):
   split='Training' if d<'2022' else 'Validation' if d<'2023' else 'Test'; a=stats[split]
   y=int(prices[s][ds[i+1]]>prices[s][d]); count=news[s,d]
   a['companies'].add(s); a['news']+=count; a['company_days']+=1; a['up']+=y; a['down_or_flat']+=1-y; a['no_news_days']+=count==0
   w.writerow([s,d,split,count,int(count>0),prices[s][d],prices[s][ds[i+1]],y])
for a in stats.values():
 a['companies']=len(a['companies']); a['up_pct']=100*a['up']/a['company_days']; a['down_or_flat_pct']=100*a['down_or_flat']/a['company_days']; assert a['up']+a['down_or_flat']==a['company_days']
with open(out/'selected_50_companies.csv','w',newline='') as f:
 w=csv.writer(f); w.writerow(['Ticker','Company','CIK','Training_Period_Unique_News'])
 for s,n in selected:w.writerow([s,ref[s]['Security'],ref[s]['CIK'],n])
with open(out/'dataset_split_statistics.csv','w',newline='') as f:
 fields=['dataset']+list(stats['Training']); w=csv.DictWriter(f,fieldnames=fields);w.writeheader()
 for split,a in stats.items():w.writerow({'dataset':split,**a})
report={'statistics':stats,'selected':selected,'expected_trading_dates_per_year':expected,'excluded_news':dict(dropped),'source_news':dict(source),'alignment':'First trading day strictly after UTC publication calendar date; conservative date-only lag','price_end':'2023-12-28','reference':'Downloaded S&P 500 snapshot from datasets/s-and-p-500-companies; historical reference date not established','sentiment':'FinBERT not run; this is a news-and-price statistics dataset, not a finished feature/model dataset'}
json.dump(report,open(out/'statistics_methodology.json','w'),ensure_ascii=False,indent=2)
print(json.dumps(report,ensure_ascii=False,indent=2),flush=True)


{
  "statistics": {
    "Training": {
      "companies": 50,
      "news": 157115,
      "company_days": 50400,
      "up": 26293,
      "down_or_flat": 24107,
      "no_news_days": 11348,
      "up_pct": 52.16865079365079,
      "down_or_flat_pct": 47.83134920634921
    },
    "Validation": {
      "companies": 50,
      "news": 39185,
      "company_days": 12550,
      "up": 6211,
      "down_or_flat": 6339,
      "no_news_days": 2683,
      "up_pct": 49.49003984063745,
      "down_or_flat_pct": 50.50996015936255
    },
    "Test": {
      "companies": 50,
      "news": 46190,
      "company_days": 12400,
      "up": 6502,
      "down_or_flat": 5898,
      "no_news_days": 2436,
      "up_pct": 52.435483870967744,
      "down_or_flat_pct": 47.564516129032256
    }
  },
  "selected": [
    [
      "INTC",
      7603
    ],
    [
      "GOOG",
      5741
    ],
    [
      "DIS",
      5649
    ],
    [
      "GILD",
      5634
    ],
    [
      "WMT",
      4976
    ],
    [
      "T"

## 独立复核

In [7]:
with open(out / 'company_day_labels.csv', newline='') as f:
    observations = list(csv.DictReader(f))
assert len(observations) == len({(r['Ticker'], r['Trading_Date']) for r in observations})
for split, expected in stats.items():
    part = [r for r in observations if r['Split'] == split]
    assert len({r['Ticker'] for r in part}) == expected['companies'] == 50
    assert len(part) == expected['company_days']
    assert sum(int(r['News_Count']) for r in part) == expected['news']
    assert sum(int(r['Label']) for r in part) == expected['up']
    assert expected['up'] + expected['down_or_flat'] == len(part)
print('复核通过：', len(observations), '条记录；', sum(int(r['News_Count']) for r in observations), '条匹配新闻')
from IPython.display import display, Markdown
lines = ['| 数据集 | 公司数 | 新闻数 | Company-Day | 上涨数量/比例 | 下跌或不变数量/比例 |', '|---|---:|---:|---:|---|---|']
for split, a in stats.items():
    lines.append(f"| {split} | {a['companies']} | {a['news']:,} | {a['company_days']:,} | {a['up']:,} / {a['up_pct']:.2f}% | {a['down_or_flat']:,} / {a['down_or_flat_pct']:.2f}% |")
display(Markdown('\n'.join(lines)))


复核通过： 75350 条记录； 242490 条匹配新闻


| 数据集 | 公司数 | 新闻数 | Company-Day | 上涨数量/比例 | 下跌或不变数量/比例 |
|---|---:|---:|---:|---|---|
| Training | 50 | 157,115 | 50,400 | 26,293 / 52.17% | 24,107 / 47.83% |
| Validation | 50 | 39,185 | 12,550 | 6,211 / 49.49% | 6,339 / 50.51% |
| Test | 50 | 46,190 | 12,400 | 6,502 / 52.44% | 5,898 / 47.56% |

## 8. 补全配置与新闻匹配

In [9]:
import hashlib, sqlite3, json, zipfile, io, bisect
import numpy as np
import pandas as pd
from pathlib import Path

FINAL = out / 'final_features'
FINAL.mkdir(exist_ok=True)
MODEL_ID = 'ProsusAI/finbert'
MODEL_REVISION = '0bc2ac10dda6929bcb1f77ccc43c7ec5c424f031'  # 下方解析并记录实际commit；后续复现使用同一commit
BATCH_SIZE = 32
MAX_LENGTH = 512
TICKERS = pd.read_csv(out / 'selected_50_companies.csv')['Ticker'].tolist()
assert len(TICKERS) == len(set(TICKERS)) == 50
base_labels = pd.read_csv(out / 'company_day_labels.csv', parse_dates=['Trading_Date'])
news_db = sqlite3.connect(base / 'news_audit.sqlite')
placeholders = ','.join('?' for _ in TICKERS)
news_items = pd.read_sql_query(
    f'SELECT ticker,date,title,url FROM news WHERE ticker IN ({placeholders})',
    news_db, params=TICKERS)
news_db.close()
news_items.columns = ['Ticker','Published_UTC','Title','URL']
news_items['Published_UTC'] = pd.to_datetime(news_items['Published_UTC'], utc=True, errors='coerce')
assert news_items['Published_UTC'].notna().all(), '存在无法解析的时间戳，应先审计'
news_items['Headline_ID'] = news_items['Title'].map(lambda s: hashlib.sha256(s.encode('utf-8')).hexdigest())
aligned = []
for ticker, frame in news_items.groupby('Ticker', sort=False):
    # days为前面价格标签单元生成的完整日期列表，包含最后一个标签价格日。
    calendar = np.array(days[ticker], dtype='datetime64[D]')
    publication_dates = frame['Published_UTC'].dt.tz_localize(None).values.astype('datetime64[D]')
    positions = np.searchsorted(calendar, publication_dates, side='right')
    valid = positions < len(calendar) - 1
    part = frame.loc[valid].copy()
    part['Trading_Date'] = pd.to_datetime(calendar[positions[valid]])
    aligned.append(part)
matched_news = pd.concat(aligned, ignore_index=True)
assert len(matched_news) == int(base_labels['News_Count'].sum())
unique_titles = matched_news[['Headline_ID','Title']].drop_duplicates('Headline_ID')
print('匹配公司—新闻记录：', len(matched_news), '；独立标题：', len(unique_titles))


匹配公司—新闻记录： 242490 ；独立标题： 176239


## 9. FinBERT批量推理与断点缓存

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from huggingface_hub import model_info

assert len(MODEL_REVISION) == 40, '必须固定已下载的模型commit'
resolved_revision = MODEL_REVISION
cache_db_path = base / f'finbert_cache_{resolved_revision[:12]}_{MAX_LENGTH}.sqlite'
cache = sqlite3.connect(cache_db_path)
cache.execute('CREATE TABLE IF NOT EXISTS scores (headline_id TEXT PRIMARY KEY, positive REAL NOT NULL, neutral REAL NOT NULL, negative REAL NOT NULL)')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=resolved_revision, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, revision=resolved_revision, local_files_only=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
torch.set_num_threads(min(4, os.cpu_count() or 1))
model.to(device).eval()
label_index = {str(v).lower(): int(k) for k,v in model.config.id2label.items()}
assert set(label_index) == {'positive','neutral','negative'}, label_index
cached_ids = {r[0] for r in cache.execute('SELECT headline_id FROM scores')}
pending = unique_titles.loc[~unique_titles['Headline_ID'].isin(cached_ids)]
print('设备：',device,'；待推理标题：',len(pending),'；模型commit：',resolved_revision)
# 按长度分批，减少padding，不改变评分定义
records = list(pending.assign(_length=pending['Title'].str.len()).sort_values('_length')[['Headline_ID','Title']].itertuples(index=False, name=None))
inference_started = time.time()
for start in range(0,len(records),BATCH_SIZE):
    chunk = records[start:start+BATCH_SIZE]
    encoded = tokenizer([r[1] for r in chunk], padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
    encoded = {k:v.to(device) for k,v in encoded.items()}
    with torch.inference_mode():
        probs = torch.softmax(model(**encoded).logits,dim=-1).cpu().numpy()
    assert np.isfinite(probs).all() and np.allclose(probs.sum(axis=1),1,atol=1e-5)
    values = [(r[0],float(p[label_index['positive']]),float(p[label_index['neutral']]),float(p[label_index['negative']])) for r,p in zip(chunk,probs)]
    cache.executemany('INSERT OR REPLACE INTO scores VALUES (?,?,?,?)',values)
    cache.commit()
    if start % (BATCH_SIZE*25) == 0:
        elapsed = time.time()-inference_started
        done = min(start+BATCH_SIZE,len(records))
        print(f'FinBERT {done:,}/{len(records):,}; {done/max(elapsed,0.001):.1f} titles/s; elapsed {elapsed/60:.1f} min', flush=True)
score_table = pd.read_sql_query('SELECT * FROM scores',cache)
cache.close()
score_table.columns = ['Headline_ID','P_Positive','P_Neutral','P_Negative']
scored_news = matched_news.merge(score_table,on='Headline_ID',how='left',validate='many_to_one')
prob_cols = ['P_Positive','P_Neutral','P_Negative']
assert scored_news[prob_cols].notna().all().all()
scored_news['Sentiment_Label'] = np.array(['Positive','Neutral','Negative'])[scored_news[prob_cols].to_numpy().argmax(axis=1)]
scored_news['Sentiment_Code'] = scored_news['Sentiment_Label'].map({'Positive':1,'Neutral':0,'Negative':-1})
scored_news.to_csv(FINAL/'news_finbert_scores.csv.gz',index=False,compression='gzip')
metadata = {'model':MODEL_ID,'revision':resolved_revision,'max_length':MAX_LENGTH,'batch_size':BATCH_SIZE,'device':str(device),'torch_threads':torch.get_num_threads(),'torch':torch.__version__}
(FINAL/'finbert_metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
print(scored_news['Sentiment_Label'].value_counts())


设备： cpu ；待推理标题： 176239 ；模型commit： 0bc2ac10dda6929bcb1f77ccc43c7ec5c424f031
FinBERT 32/176,239; 30.1 titles/s; elapsed 0.0 min
FinBERT 832/176,239; 306.8 titles/s; elapsed 0.0 min
FinBERT 1,632/176,239; 351.9 titles/s; elapsed 0.1 min
FinBERT 2,432/176,239; 359.1 titles/s; elapsed 0.1 min
FinBERT 3,232/176,239; 363.6 titles/s; elapsed 0.1 min
FinBERT 4,032/176,239; 365.5 titles/s; elapsed 0.2 min
FinBERT 4,832/176,239; 368.5 titles/s; elapsed 0.2 min
FinBERT 5,632/176,239; 369.6 titles/s; elapsed 0.3 min
FinBERT 6,432/176,239; 370.5 titles/s; elapsed 0.3 min
FinBERT 7,232/176,239; 372.8 titles/s; elapsed 0.3 min
FinBERT 8,032/176,239; 371.3 titles/s; elapsed 0.4 min
FinBERT 8,832/176,239; 370.6 titles/s; elapsed 0.4 min
FinBERT 9,632/176,239; 369.4 titles/s; elapsed 0.4 min
FinBERT 10,432/176,239; 368.8 titles/s; elapsed 0.5 min
FinBERT 11,232/176,239; 365.9 titles/s; elapsed 0.5 min
FinBERT 12,032/176,239; 363.8 titles/s; elapsed 0.6 min
FinBERT 12,832/176,239; 360.5 titles/s; elapsed 

## 10. Company-Day情感特征

In [11]:
scored_news['is_positive'] = (scored_news['Sentiment_Label']=='Positive').astype(int)
scored_news['is_negative'] = (scored_news['Sentiment_Label']=='Negative').astype(int)
daily_sentiment = scored_news.groupby(['Ticker','Trading_Date']).agg(
    Scored_News_Count=('Sentiment_Code','size'),
    Positive_Count=('is_positive','sum'), Negative_Count=('is_negative','sum'),
    Average_Sentiment_Score=('Sentiment_Code','mean')).reset_index()
daily_sentiment['Positive_Ratio'] = daily_sentiment['Positive_Count']/daily_sentiment['Scored_News_Count']
daily_sentiment['Negative_Ratio'] = daily_sentiment['Negative_Count']/daily_sentiment['Scored_News_Count']
features = base_labels.merge(daily_sentiment,on=['Ticker','Trading_Date'],how='left',validate='one_to_one')
fill_cols = ['Scored_News_Count','Positive_Count','Negative_Count','Positive_Ratio','Negative_Ratio','Average_Sentiment_Score']
features[fill_cols] = features[fill_cols].fillna(0)
assert (features['News_Count']==features['Scored_News_Count']).all()
assert np.allclose(features['Average_Sentiment_Score'],features['Positive_Ratio']-features['Negative_Ratio'])
assert (features['HasNews']==(features['News_Count']>0).astype(int)).all()


## 11. 市场特征及标签日期

In [12]:
market_parts = []
with zipfile.ZipFile(base/'full_history.zip') as archive:
    for ticker in TICKERS:
        with archive.open(f'full_history/{ticker}.csv') as stream:
            price = pd.read_csv(stream,parse_dates=['date'])
        price = price.sort_values('date').copy()
        assert not price['date'].duplicated().any(), f'{ticker}存在重复日期'
        for column in ['adj close','volume']:
            price[column] = pd.to_numeric(price[column],errors='coerce')
        price.loc[~np.isfinite(price['adj close']) | (price['adj close']<=0),'adj close'] = np.nan
        price.loc[~np.isfinite(price['volume']) | (price['volume']<0),'volume'] = np.nan
        price['Return'] = price['adj close']/price['adj close'].shift(1)-1
        previous_volume = price['volume'].shift(1).where(lambda x:x>0)
        price['Volume_Change'] = price['volume']/previous_volume-1
        price['Volatility_5D'] = price['Return'].rolling(5,min_periods=5).std(ddof=1)
        price['Next_Trading_Date'] = price['date'].shift(-1)
        price['Next_Return'] = price['adj close'].shift(-1)/price['adj close']-1
        price['Ticker'] = ticker
        price = price.rename(columns={'date':'Trading_Date','volume':'Volume'})
        market_parts.append(price[['Ticker','Trading_Date','Volume','Return','Volume_Change','Volatility_5D','Next_Trading_Date','Next_Return']])
market = pd.concat(market_parts,ignore_index=True)
features = features.merge(market,on=['Ticker','Trading_Date'],how='left',validate='one_to_one')
for column in ['Adjusted_Close','Next_Adjusted_Close']:
    features[column] = pd.to_numeric(features[column],errors='coerce')
assert np.allclose(features['Next_Return'],features['Next_Adjusted_Close']/features['Adjusted_Close']-1,equal_nan=False)
assert (features['Label']==(features['Next_Return']>0).astype(int)).all()
print(features[['Return','Volume_Change','Volatility_5D']].describe())


             Return  Volume_Change  Volatility_5D
count  75350.000000   75349.000000   75350.000000
mean       0.000645       0.278671       0.018333
std        0.030615      56.177333       0.024690
min       -0.948942      -1.000000       0.000453
25%       -0.009467      -0.193638       0.009641
50%        0.000670      -0.012440       0.014501
75%        0.010683       0.222336       0.021937
max        5.326960   15420.000000       2.387842


## 12. 严格时间边界与共同有效样本

In [13]:
SENTIMENT_FEATURES = ['Positive_Ratio','Negative_Ratio','News_Count','HasNews']
MARKET_FEATURES = ['Return','Volume_Change','Volatility_5D']
FEATURE_SETS = {'Sentiment-only':SENTIMENT_FEATURES,'Market-only':MARKET_FEATURES,'Combined':SENTIMENT_FEATURES+MARKET_FEATURES}
def split_for_dates(values):
    return pd.Series(np.select([values.between('2018-01-01','2021-12-31'),values.between('2022-01-01','2022-12-31'),values.between('2023-01-01','2023-12-31')],['Training','Validation','Test'],default='Outside'),index=values.index)
assert (features['Split']==split_for_dates(features['Trading_Date'])).all()
features['Label_Split'] = split_for_dates(features['Next_Trading_Date'])
features['Cross_Split_Boundary'] = features['Split']!=features['Label_Split']
numeric_predictors = features[SENTIMENT_FEATURES+MARKET_FEATURES].apply(pd.to_numeric,errors='coerce')
features['Invalid_Features'] = ~np.isfinite(numeric_predictors.to_numpy()).all(axis=1)
features['Invalid_Label'] = features['Next_Trading_Date'].isna() | ~np.isfinite(features['Next_Return'])
features['Included'] = ~(features['Cross_Split_Boundary'] | features['Invalid_Features'] | features['Invalid_Label'])
features.to_csv(FINAL/'company_day_features_all.csv',index=False)
excluded = features.loc[~features['Included']].copy()
excluded.to_csv(FINAL/'excluded_company_days.csv',index=False)
model_data = features.loc[features['Included']].sort_values(['Trading_Date','Ticker']).reset_index(drop=True)
assert not model_data.duplicated(['Ticker','Trading_Date']).any()
model_data.to_csv(FINAL/'company_day_features.csv',index=False)
for split in ['Training','Validation','Test']:
    model_data.loc[model_data['Split']==split].to_csv(FINAL/f'{split.lower()}_features.csv',index=False)
(FINAL/'feature_sets.json').write_text(json.dumps(FEATURE_SETS,indent=2),encoding='utf-8')
print('基础样本：',len(features),'；排除：',len(excluded),'；最终共同样本：',len(model_data))
print(excluded.groupby('Split')[['Cross_Split_Boundary','Invalid_Features','Invalid_Label']].sum())


基础样本： 75350 ；排除： 101 ；最终共同样本： 75249
            Cross_Split_Boundary  Invalid_Features  Invalid_Label
Split                                                            
Test                           0                 1              0
Training                      50                 0              0
Validation                    50                 0              0


## 13. 最终2.4统计与EDA中间表

In [14]:
final_stats = model_data.groupby('Split').agg(Companies=('Ticker','nunique'),News=('News_Count','sum'),Company_Days=('Label','size'),Up=('Label','sum')).reindex(['Training','Validation','Test'])
final_stats['Down_or_Flat'] = final_stats['Company_Days']-final_stats['Up']
final_stats['Up_Pct'] = 100*final_stats['Up']/final_stats['Company_Days']
final_stats['Down_or_Flat_Pct'] = 100-final_stats['Up_Pct']
assert (final_stats['Up']+final_stats['Down_or_Flat']==final_stats['Company_Days']).all()
final_stats.to_csv(FINAL/'final_dataset_statistics.csv',encoding='utf-8-sig')
model_data['Sentiment_Group'] = np.select([model_data['HasNews']==0, model_data['Average_Sentiment_Score']>0, model_data['Average_Sentiment_Score']<0],['NoNews','Positive','Negative'],default='Neutral')
eda_groups = model_data.groupby(['Split','Sentiment_Group']).agg(Company_Days=('Label','size'),Mean_Next_Return=('Next_Return','mean'),Up_Ratio=('Label','mean'))
eda_groups.to_csv(FINAL/'eda_sentiment_groups.csv',encoding='utf-8-sig')
scored_news['Sentiment_Label'].value_counts().to_csv(FINAL/'eda_news_sentiment_distribution.csv')
model_data.groupby('Split')['News_Count'].describe().to_csv(FINAL/'eda_news_count_distribution.csv')
print(final_stats.to_string())
print(eda_groups.to_string())


            Companies    News  Company_Days     Up  Down_or_Flat     Up_Pct  Down_or_Flat_Pct
Split                                                                                        
Training           50  156997         50350  26258         24092  52.150943         47.849057
Validation         50   39087         12500   6183          6317  49.464000         50.536000
Test               50   46186         12399   6502          5897  52.439713         47.560287
                            Company_Days  Mean_Next_Return  Up_Ratio
Split      Sentiment_Group                                          
Test       Negative                 2199          0.001174  0.519327
           Neutral                  4480          0.001145  0.518527
           NoNews                   2436          0.000665  0.527915
           Positive                 3284          0.000854  0.533191
Training   Negative                 8650          0.000107  0.511561
           Neutral                 17171       

## 数据质量提示

In [16]:
review = pd.read_csv(FINAL/'company_day_features.csv')
review = review.loc[(review['Return'].abs()>0.5) | (review['Volume_Change'].abs()>100), ['Ticker','Trading_Date','Return','Volume_Change','Volatility_5D']]
review.to_csv(FINAL/'market_extreme_values_for_review.csv',index=False)
print(review.to_string(index=False))


Ticker Trading_Date    Return  Volume_Change  Volatility_5D
   PCG   2019-01-14 -0.523593       5.440217       0.228611
   PCG   2019-01-24  0.745932       2.815741       0.328407
   OXY   2020-03-09 -0.520138       1.692654       0.219527
   NEE   2020-04-02 -0.761399       1.697908       0.339362
    GE   2020-07-06  5.326960      -0.837019       2.376530
  GOOG   2020-07-06 -0.948942      15.825346       0.432785
  BIIB   2020-11-09 -0.281666   15420.000000       0.263294


## 最终清理与交付

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

SOURCE = Path('outputs/final_features')
DEST = Path('outputs/final_cleaned')
DEST.mkdir(exist_ok=True)
original = pd.read_csv(SOURCE/'company_day_features.csv',parse_dates=['Trading_Date','Next_Trading_Date'])
breakpoints = [('GOOG','2020-07-06'),('GE','2020-07-06'),('NEE','2020-04-02')]
reasons = {}
for ticker, date_text in breakpoints:
    date = pd.Timestamp(date_text)
    company = original.loc[original['Ticker']==ticker].sort_values('Trading_Date')
    positions = np.flatnonzero(company['Trading_Date'].eq(date).to_numpy())
    assert len(positions)==1
    position = int(positions[0])
    # 回溯至完整研究期的原始交易日网格，保证按交易日而非自然日计算5日窗口。
    full = pd.read_csv('outputs/company_day_labels.csv',parse_dates=['Trading_Date'])
    calendar = full.loc[full['Ticker']==ticker,'Trading_Date'].sort_values().tolist()
    start = calendar.index(date)
    affected_dates = calendar[start:start+5]
    for idx in company.index[company['Trading_Date'].isin(affected_dates)]:
        reasons.setdefault(idx,[]).append(f'{ticker}_{date_text}:feature_window_crosses_price_break')
    for idx in company.index[company['Next_Trading_Date'].eq(date)]:
        reasons.setdefault(idx,[]).append(f'{ticker}_{date_text}:target_return_crosses_price_break')
assert len(reasons)==18, len(reasons)
removed = original.loc[sorted(reasons)].copy()
removed['Exclusion_Reason'] = [ '; '.join(reasons[i]) for i in removed.index]
removed.to_csv(DEST/'excluded_price_break_records.csv',index=False,encoding='utf-8-sig')
cleaned = original.drop(index=list(reasons)).sort_values(['Trading_Date','Ticker']).reset_index(drop=True)
assert len(cleaned)==75231
assert not cleaned.duplicated(['Ticker','Trading_Date']).any()
assert cleaned['Ticker'].nunique()==50
feature_sets = json.loads((SOURCE/'feature_sets.json').read_text())
assert np.isfinite(cleaned[feature_sets['Combined']].to_numpy()).all()
assert (cleaned['Label']==(cleaned['Next_Adjusted_Close']>cleaned['Adjusted_Close']).astype(int)).all()
assert np.allclose(cleaned['Average_Sentiment_Score'],cleaned['Positive_Ratio']-cleaned['Negative_Ratio'])
assert not cleaned['Cross_Split_Boundary'].any()
cleaned.to_csv(DEST/'company_day_features_final.csv',index=False,encoding='utf-8-sig')
for split in ['Training','Validation','Test']:
    cleaned.loc[cleaned['Split']==split].to_csv(DEST/f'{split.lower()}_features_final.csv',index=False,encoding='utf-8-sig')
    if split!='Training':
        before=original.loc[original['Split']==split].reset_index(drop=True)
        after=cleaned.loc[cleaned['Split']==split].reset_index(drop=True)
        pd.testing.assert_frame_equal(before,after)
earlier_excluded=pd.read_csv(SOURCE/'excluded_company_days.csv')
earlier_excluded['Exclusion_Reason']='Earlier boundary/invalid-feature exclusion; see boolean columns'
pd.concat([earlier_excluded,removed],ignore_index=True).to_csv(DEST/'all_excluded_records.csv',index=False,encoding='utf-8-sig')
print('原共同样本：',len(original),'新增排除：',len(removed),'最终共同样本：',len(cleaned))
print(removed.groupby('Ticker').size())


原共同样本： 75249 新增排除： 18 最终共同样本： 75231
Ticker
GE      6
GOOG    6
NEE     6
dtype: int64


In [11]:
summary = cleaned.groupby('Split').agg(Companies=('Ticker','nunique'),News=('News_Count','sum'),Company_Days=('Label','size'),Up=('Label','sum')).reindex(['Training','Validation','Test'])
summary['Down_or_Flat'] = summary['Company_Days']-summary['Up']
summary['Up_Pct'] = 100*summary['Up']/summary['Company_Days']
summary['Down_or_Flat_Pct'] = 100-summary['Up_Pct']
summary.to_csv(DEST/'final_dataset_statistics.csv',encoding='utf-8-sig')
news_scores = pd.read_csv(SOURCE/'news_finbert_scores.csv.gz',parse_dates=['Trading_Date'])
final_news = news_scores.merge(cleaned[['Ticker','Trading_Date','Split']],on=['Ticker','Trading_Date'],how='inner',validate='many_to_one')
assert len(final_news)==cleaned['News_Count'].sum()
actual_counts=final_news.groupby(['Ticker','Trading_Date']).size().rename('count')
check=cleaned.join(actual_counts,on=['Ticker','Trading_Date'])
assert (check['count'].fillna(0)==check['News_Count']).all()
final_news.to_csv(DEST/'news_finbert_scores_final.csv.gz',index=False,compression='gzip')
final_news.groupby(['Split','Sentiment_Label']).size().to_csv(DEST/'eda_news_sentiment_distribution.csv')
eda = cleaned.copy()
eda['Sentiment_Group']=np.select([eda['HasNews']==0,eda['Average_Sentiment_Score']>0,eda['Average_Sentiment_Score']<0],['NoNews','Positive','Negative'],default='Neutral')
eda.groupby(['Split','Sentiment_Group']).agg(Company_Days=('Label','size'),Mean_Next_Return=('Next_Return','mean'),Up_Ratio=('Label','mean')).to_csv(DEST/'eda_sentiment_groups.csv')
cleaned.groupby('Split')['News_Count'].describe().to_csv(DEST/'eda_news_count_distribution.csv')
for file in ['feature_sets.json','finbert_metadata.json']:
    (DEST/file).write_bytes((SOURCE/file).read_bytes())
report={'status':'passed','base_rows':75350,'earlier_excluded':len(earlier_excluded),'price_break_excluded':len(removed),'final_rows':len(cleaned),'companies':50,'final_matched_news':len(final_news),'breakpoints':breakpoints,'retained_large_market_moves':'Other large moves retained; not all large values are data errors'}
(DEST/'verification.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
assert report['base_rows']-report['earlier_excluded']-report['price_break_excluded']==len(cleaned)
print(summary.to_string())
print('新闻逐日对账、特征有限值、标签、共同样本及验证/测试未改变检查通过。')


            Companies    News  Company_Days     Up  Down_or_Flat     Up_Pct  Down_or_Flat_Pct
Split                                                                                        
Training           50  156929         50332  26248         24084  52.149726         47.850274
Validation         50   39087         12500   6183          6317  49.464000         50.536000
Test               50   46186         12399   6502          5897  52.439713         47.560287
新闻逐日对账、特征有限值、标签、共同样本及验证/测试未改变检查通过。
